<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); border-radius: 16px; padding: 36px 40px; margin-bottom: 8px;">
  <h1 style="color: #e0aaff; font-size: 2.4em; font-weight: 800; margin: 0 0 10px 0; letter-spacing: 1px;">
    🧊 iTransformer · Stage 1 — Preprocessing
  </h1>
  <p style="color: #c77dff; font-size: 1.15em; margin: 0 0 18px 0; font-weight: 500;">
    Raw parquet → aligned master grid → engineered features → one frozen, hash-verified artifact
  </p>
  <hr style="border: none; border-top: 1px solid #7b2d8b; margin: 16px 0;">
  <p style="color: #9d8cff; font-size: 0.97em; margin: 0;">
    This notebook does every part of the pipeline that <strong>does not need a GPU</strong>: load, validate, resolve the gold timezone, apply publication lags, align four sampling frequencies onto the BTC minute grid, engineer features, and fit train-only statistics. It ends by writing a frozen artifact to <code>data/processed/features_&lt;profile&gt;/</code> together with the hashes that bind it to the raw files it came from. <strong>Run it on your own machine</strong> — it costs nothing and consumes no Kaggle GPU quota. Then run <code>02_train.ipynb</code>, which consumes the artifact and never opens the raw data at all.
  </p>
</div>

<div style="background: linear-gradient(90deg, #10002b, #240046); border-left: 4px solid #c77dff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">🗂️ How to run this</h2>
  <p style="color: #9d8cff; margin: 0 0 8px 0;">Designed for a <strong>local machine</strong>, not a Kaggle session. Preprocessing is CPU-bound and it is the memory ceiling of the whole project, so it belongs where RAM is free and GPU quota is not being spent. A Kaggle CPU session works as a fallback.</p>  <ul style="color: #9d8cff; margin: 0; padding-left: 20px;">
    <li><strong>Local run.</strong> Working directory is <code>notebooks/</code>; the raw files are found at <code>../data/raw</code> automatically.</li>
    <li><strong>Pick a profile.</strong> <code>'tiny'</code> = three months, runs in minutes and exists to verify the notebook itself. <code>'smoke'</code> = six months. <code>'full'</code> = the whole 2018–2026 grid.</li>
    <li><strong>One artifact per profile.</strong> <code>data/processed/features_&lt;profile&gt;/</code> is keyed by profile, so a <code>tiny</code> artifact can never be mistaken for a <code>full</code> one.</li>
    <li><strong>Publish once.</strong> Upload the <code>full</code> artifact as a Kaggle Dataset and every training session afterwards reads it instead of rebuilding it.</li>
    <li><strong>Re-run only when features change.</strong> Model, optimiser and evaluation knobs live in <code>02_train.ipynb</code> and never require touching this notebook.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #10002b, #240046); border-left: 4px solid #c77dff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">🧭 Notebook map</h2>
  <p style="color: #9d8cff; margin: 0 0 8px 0;">Sections §1–§8 are unchanged from the original single notebook; §9–§17 moved to <code>02_train.ipynb</code>. The three closing sections are new and exist only because the pipeline is now split.</p>  <ul style="color: #9d8cff; margin: 0; padding-left: 20px;">
    <li><strong>§1 Setup</strong> — imports, device detection, seeding, plot theme</li>
    <li><strong>§2 Configuration</strong> — one <code>CFG</code> object; profiles; publication-lag table</li>
    <li><strong>§3 Loading</strong> — typed loaders, string→float casting, UTC normalisation</li>
    <li><strong>§4 Validation</strong> — gap census, OHLC sanity, extreme-return census</li>
    <li><strong>§5 Gold timezone</strong> — resolved empirically, not assumed</li>
    <li><strong>§6 Alignment</strong> — publication lags, backward as-of joins, staleness features</li>
    <li><strong>§7 Features</strong> — price · volume · cross-asset · macro · temporal blocks</li>
    <li><strong>§8 Hygiene</strong> — train-only winsorisation, collinearity pruning, scaler</li>
    <li><strong>Causality gate</strong> — shift equivariance and future-perturbation invariance</li>
    <li><strong>Freeze</strong> — six artifact files plus the hashes that tie them to the raw data</li>
    <li><strong>Verify</strong> — the artifact re-read through the consumer's own rejection rules</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #10002b, #240046); border-left: 4px solid #c77dff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">📦 §1 · Setup &amp; Imports</h2>
  <p style="color: #9d8cff; margin: 0 0 8px 0;">PyTorch is the <strong>only</strong> deep-learning framework here — no TensorFlow, Keras, or JAX. Polars does the heavy dataframe work over the 4.4M-row minute table; NumPy holds the final feature matrix.</p>
</div>

In [2]:
import copy
import gc
import hashlib
import json
import math
import os
import random
import subprocess
import sys
import time
import warnings
from dataclasses import asdict, dataclass, field
from datetime import datetime, timedelta, timezone
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="torch")


def _ensure(pkg: str, import_name: str | None = None) -> None:
    """Install a package only if it is genuinely missing (Kaggle ships most of these)."""
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        print(f"[setup] installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)


_ensure("polars")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

print(f"python  {sys.version.split()[0]}")
print(f"torch   {torch.__version__}")
print(f"polars  {pl.__version__}")
print(f"numpy   {np.__version__}")


def peak_rss_gb() -> float:
    """Peak resident set size of this process, in GiB. Stdlib only, no extra dependency.

    Preprocessing is the memory ceiling of this project and the local machine has *less* RAM
    than Kaggle does, so the number has to be measured rather than assumed.
    """
    try:
        if sys.platform == "win32":
            import ctypes
            from ctypes import wintypes

            class _PMC(ctypes.Structure):
                _fields_ = [("cb", wintypes.DWORD), ("PageFaultCount", wintypes.DWORD),
                            ("PeakWorkingSetSize", ctypes.c_size_t),
                            ("WorkingSetSize", ctypes.c_size_t),
                            ("QuotaPeakPagedPoolUsage", ctypes.c_size_t),
                            ("QuotaPagedPoolUsage", ctypes.c_size_t),
                            ("QuotaPeakNonPagedPoolUsage", ctypes.c_size_t),
                            ("QuotaNonPagedPoolUsage", ctypes.c_size_t),
                            ("PagefileUsage", ctypes.c_size_t),
                            ("PeakPagefileUsage", ctypes.c_size_t)]

            pmc = _PMC()
            pmc.cb = ctypes.sizeof(_PMC)
            # c_void_p(-1) is the current-process pseudo-handle. Calling GetCurrentProcess()
            # through ctypes returns c_int, which truncates -1 to 32 bits on a 64-bit build
            # and makes the call fail silently, reporting 0 GB.
            ok = ctypes.windll.psapi.GetProcessMemoryInfo(
                ctypes.c_void_p(-1), ctypes.byref(pmc), pmc.cb)
            return pmc.PeakWorkingSetSize / 1024**3 if ok else float("nan")
        with open("/proc/self/status") as fh:                    # Linux, including Kaggle
            for line in fh:
                if line.startswith("VmHWM:"):
                    return int(line.split()[1]) / 1024**2
    except Exception:
        pass
    return float("nan")

python  3.14.5
torch   2.13.0+cpu
polars  1.43.1
numpy   2.5.1


<div style="background: linear-gradient(90deg, #10002b, #240046); border-left: 4px solid #c77dff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">🖥️ Device, precision, and reproducibility</h2>
  <p style="color: #9d8cff; margin: 0 0 8px 0;">Three things are decided here, and each one has a wrong answer that silently costs you:</p>  <ul style="color: #9d8cff; margin: 0; padding-left: 20px;">
    <li><strong>Device</strong> — never hard-coded to <code>.cuda()</code>; the notebook runs on CPU too (slowly), which matters for debugging.</li>
    <li><strong>AMP dtype</strong> — <code>bfloat16</code> needs compute capability ≥ 8.0. <strong>Kaggle's T4 is Turing (sm_75) and does not support it</strong>, so the notebook detects this and falls back to <code>float16</code> + <code>GradScaler</code>. Forcing bf16 on a T4 is emulated and slower than fp32.</li>
    <li><strong>Seeding</strong> — <code>random</code>, <code>numpy</code>, <code>torch</code>, CUDA, <code>PYTHONHASHSEED</code>, and DataLoader workers all seeded. <code>cudnn.deterministic</code> is switched on for the final run and off while exploring, because determinism costs throughput.</li>
  </ul>
</div>

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()

# bfloat16 needs sm_80+. Kaggle's T4 is sm_75 -> fp16 + GradScaler instead.
if DEVICE.type == "cuda":
    # torch.cuda.is_bf16_supported() defaults to including_emulation=True and returns
    # True on Turing (sm_75), because a bf16 *tensor* can be allocated there even though
    # there is no bf16 tensor-core path. Trusting it puts Kaggle's T4 on EMULATED bf16,
    # which is slower than plain fp32. Ask the hardware directly instead.
    BF16_OK = torch.cuda.get_device_capability(0)[0] >= 8
    AMP_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
    USE_SCALER = not BF16_OK              # GradScaler is only needed for fp16
    for i in range(N_GPU):
        prop = torch.cuda.get_device_properties(i)
        print(f"gpu[{i}] {prop.name}  sm_{prop.major}{prop.minor}  {prop.total_memory / 1024**3:.1f} GB")
else:
    AMP_DTYPE, USE_SCALER = torch.float32, False

AMP_ENABLED = DEVICE.type == "cuda"
print(f"\ndevice      {DEVICE}  (n_gpu={N_GPU})")
print(f"amp dtype   {AMP_DTYPE}   grad_scaler={USE_SCALER}")


def set_seed(seed: int, deterministic: bool = False) -> None:
    """Seed every source of randomness the training loop touches."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic


def seed_worker(worker_id: int) -> None:
    """DataLoader worker_init_fn - workers inherit a derived, reproducible seed."""
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)

<div style="background: linear-gradient(90deg, #10002b, #240046); border-left: 4px solid #c77dff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">🎨 Plot theme</h2>
  <p style="color: #9d8cff; margin: 0 0 8px 0;">One palette, defined once, used everywhere. The five categorical hues are a <strong>fixed order that is never cycled</strong> and were validated for colour-vision deficiency against this dark surface — worst adjacent pair ΔE 15.9 (deutan), well above the ΔE 8 target. Magnitude uses a single-hue ramp, polarity uses a two-hue diverging ramp with a neutral grey midpoint, and no chart in this notebook uses two y-axes.</p>
</div>

In [ ]:
SURFACE = "#1a1a19"
INK = "#e8e6f0"
INK_MUTED = "#9d97b5"
GRID = "#2e2c3d"

# Fixed categorical order - assigned by entity, never by rank, never cycled.
CAT = ["#2f9e68", "#a855f7", "#cf7400", "#1e9ec4", "#e94560"]
SEQ = ["#2b1a3d", "#4c2b6b", "#7040a0", "#9a5fd0", "#c391f0"]   # magnitude: one hue, light->dark
DIV = ["#1e9ec4", "#7fa8b8", "#8a8a8a", "#c78591", "#e94560"]   # polarity: two hues + grey midpoint

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.edgecolor": GRID, "grid.color": GRID, "grid.linewidth": 0.6, "grid.alpha": 0.6,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "legend.labelcolor": INK,
    "lines.linewidth": 2.0, "lines.markersize": 5,
    "figure.dpi": 110, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
})


def finish(ax, title: str = "", xlabel: str = "", ylabel: str = "", legend: bool = False):
    """Consistent titling. Legend appears whenever 2+ series share an axis."""
    if title:
        ax.set_title(title, loc="left", pad=12)
    ax.set_xlabel(xlabel, color=INK_MUTED)
    ax.set_ylabel(ylabel, color=INK_MUTED)
    if legend:
        ax.legend(loc="best", fontsize=9)
    return ax

<div style="background: linear-gradient(90deg, #1a1a2e, #16213e); border-left: 4px solid #e94560; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #f5a623; margin: 0 0 10px 0;">🎛️ §2 · Configuration</h2>
  <p style="color: #ffd6a5; margin: 0 0 8px 0;">Every knob lives in this one cell. No magic numbers appear later in the notebook — if a number matters, it is named here and referenced by name.</p>  <ul style="color: #ffd6a5; margin: 0; padding-left: 20px;">
    <li><strong><code>PROFILE</code></strong> rescales the whole notebook. <code>'smoke'</code> = six months, stride 60, 2 epochs, small model (~15 min). <code>'full'</code> = everything, stride 5, 30 epochs, <code>d_model=512</code> (~8–10 h).</li>
    <li><strong><code>RAW_DIR</code></strong> is the one path you must edit for your Kaggle Dataset mount.</li>
    <li><strong>Exogenous history is always loaded in full</strong>, regardless of profile. Only the BTC minute grid is shortened. This matters: a 36-month macro z-score needs 36 months of macro history even when you are training on six months of minutes.</li>
    <li><strong>The publication-lag table</strong> is part of the config, not buried in code, because it is the single most consequential anti-leakage decision in the project.</li>
  </ul>
</div>

In [ ]:
# ==========================================================================
#  EDIT THIS ONE LINE to match your Kaggle Dataset mount path.
#  Local runs fall back to ../data/raw automatically.
# ==========================================================================
KAGGLE_RAW_DIR = Path("/kaggle/input/itransformer-btc-raw")

# Resuming across Kaggle sessions: attach the PREVIOUS session's output as an input
# dataset and point this at the checkpoint folder inside it, e.g.
#   "/kaggle/input/itransformer-session-1/checkpoints/full_L1440_H60_d512_s42"
# Leave as None for a fresh run. See docs/KAGGLE_GUIDE.md.
KAGGLE_RESUME_DIR: str | None = None

PROFILE = "smoke"          # 'tiny' (~2 min, CPU) | 'smoke' (~15 min) | 'full' (staged, see guide)

REQUIRED_FILES = (
    [f"btc_usdt_binance_{y}.parquet" for y in range(2018, 2027)]
    + ["xauusd_2018_2026.parquet",
       "US_Dollar_Index_2018_2026.parquet",
       "fed_economic_data_2018_2026.parquet"]
)

ON_KAGGLE = Path("/kaggle/input").exists()


def _complete(d: Path) -> bool:
    return d.is_dir() and not [f for f in REQUIRED_FILES if not (d / f).exists()]


def _discover_raw_dir() -> Path:
    """Locate the folder holding all 12 raw files.

    Kaggle mounts a dataset at /kaggle/input/<slug>, and the slug is whatever the
    uploader named it - hard-coding one guarantees a broken first run for everyone
    else. Search the mount points instead, one level deep for zipped-folder uploads.
    """
    cands = [KAGGLE_RAW_DIR, Path("../data/raw"), Path("data/raw")]
    root = Path("/kaggle/input")
    if root.exists():
        tops = sorted(p for p in root.iterdir() if p.is_dir())
        cands += tops + [q for p in tops for q in sorted(p.iterdir()) if q.is_dir()]
    return next((c for c in cands if _complete(c)), KAGGLE_RAW_DIR)


RAW_DIR = _discover_raw_dir()
WORK_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../artifacts")
WORK_DIR.mkdir(parents=True, exist_ok=True)

_missing = [f for f in REQUIRED_FILES if not (RAW_DIR / f).exists()]
assert not _missing, (
    f"RAW_DIR={RAW_DIR} is missing {len(_missing)} file(s): {_missing[:4]}...\n"
    f"Attach the dataset holding the 12 raw parquet files, or set KAGGLE_RAW_DIR at "
    f"the top of this cell to its mount path."
)
print(f"raw   -> {RAW_DIR}   ({len(REQUIRED_FILES)} files present)")
print(f"work  -> {WORK_DIR}")

In [ ]:
@dataclass
class Config:
    """Single source of truth. Serialised verbatim into the export bundle."""

    # ---- identity -------------------------------------------------------
    profile: str = PROFILE
    seed: int = 42
    run_id: str = ""

    # ---- master grid ----------------------------------------------------
    grid_start: str = "2018-01-02 00:00"
    grid_end: str = "2026-05-31 23:59"     # gold/macro end here; BTC's extra June 2026 is discarded
    # Gold broker-clock offset in hours, resolved empirically in section 5.
    # 0 = the file is already UTC. 'auto' re-derives it from the data.
    gold_utc_offset_h: object = 0

    # ---- windowing ------------------------------------------------------
    seq_len: int = 1440                    # L: one full day of minutes
    pred_len: int = 60                     # H: predict the next 60 one-minute log returns
    train_stride: int = 5                  # eval strides are always 1
    horizons: tuple = (1, 5, 15, 30, 60)   # cumulative horizons reported downstream

    # ---- data quality tolerances ----------------------------------------
    max_synth_run_in_window: int = 15      # reject a window containing >15 min of synthetic BTC bars
    gold_staleness_cap_min: int = 4320     # 3 days: a normal weekend is fine, an outage is not

    # ---- splits ---------------------------------------------------------
    train_end: str = "2023-12-31 23:59"
    val_end: str = "2024-12-31 23:59"
    test_end: str = "2026-05-31 23:59"
    embargo_safety_min: int = 1440         # extra margin on top of L + H

    # ---- model ----------------------------------------------------------
    d_model: int = 512
    n_heads: int = 8
    e_layers: int = 3
    d_ff: int = 2048
    dropout: float = 0.1
    activation: str = "gelu"
    use_norm: bool = True                  # RevIN-style instance normalisation
    use_linear_skip: bool = True           # DLinear-style residual skip, zero-initialised
    project_target_only: bool = True
    norm_style: str = "post"               # 'post' reproduces thuml/iTransformer; 'pre' deviates
    patch_len: int = 60                    # TimeXer / PatchTST patch length, in minutes

    # ---- optimisation ---------------------------------------------------
    epochs: int = 30
    batch_size: int = 128
    lr: float = 3e-4
    weight_decay: float = 1e-4
    betas: tuple = (0.9, 0.98)
    warmup_frac: float = 0.05
    grad_clip: float = 1.0
    loss: str = "huber"                    # 'mse' | 'huber' | 'pinball' | 'huber_dir'
    huber_delta: float = 1.0
    dir_lambda: float = 0.1                # only used by 'huber_dir'
    quantiles: tuple = (0.1, 0.5, 0.9)     # only used by 'pinball'
    early_stop_patience: int = 5
    num_workers: int = 2
    deterministic: bool = False

    # ---- what to run ----------------------------------------------------
    # Stage switches. One 12 h Kaggle session cannot fit the main model plus six
    # trained baselines plus five ablations plus walk-forward at the full profile -
    # run them in separate sessions and let the checkpoints carry over.
    run_baselines: bool = True             # AR / DLinear / PatchTST / TimeXer / VanillaTF
    run_vanilla_transformer: bool = True   # the expensive baseline (L x L attention)
    run_ablation: bool = True
    run_walkforward: bool = False          # ~5x runtime; see section 15
    walkforward_months: int = 3
    resume: bool = True
    # Stop training while there is still session left, so the checkpoint is written and
    # the version can be saved. Kaggle kills a session at the 12 h wall without warning.
    session_budget_hours: float = 11.0     # 0 disables the guard
    reserve_hours: float = 0.5             # stop this long before the budget runs out

    # ---- evaluation / backtest ------------------------------------------
    eval_max_windows: int = 200_000        # cap eval windows so a full pass stays tractable
    backtest_horizon: int = 60
    fee_per_side: float = 0.0004           # Binance spot taker, 4 bps
    slippage_per_side: float = 0.0002      # 2 bps, conservative for BTC/USDT at 1 min
    dir_acc_eps_bp: float = 1.0            # ignore |return| < 1 bp when scoring direction
    n_seeds_report: int = 5

    # ---- feature blocks (toggles feed the ablation table) ---------------
    blocks: tuple = ("btc_price", "btc_volume", "btc_momentum",
                     "gold", "cross_asset", "dxy", "macro", "temporal")
    macro_n_pca: int = 4
    fracdiff_grid: tuple = (0.2, 0.3, 0.4, 0.5, 0.6)
    fracdiff_width: int = 512
    winsor_q: float = 0.001
    collinear_thresh: float = 0.98

    @property
    def embargo_min(self) -> int:
        return self.seq_len + self.pred_len + self.embargo_safety_min

    @property
    def split_gap_min(self) -> int:
        """purge (= H) + embargo, in minutes, inserted at every split boundary."""
        return self.pred_len + self.embargo_min


CFG = Config()

# ---- profile overrides ---------------------------------------------------
if CFG.profile == "smoke":
    CFG.grid_start, CFG.grid_end = "2021-01-02 00:00", "2021-06-30 23:59"
    CFG.train_end, CFG.val_end, CFG.test_end = (
        "2021-04-30 23:59", "2021-05-31 23:59", "2021-06-30 23:59")
    CFG.seq_len, CFG.pred_len = 480, 60
    CFG.train_stride = 60
    CFG.d_model, CFG.d_ff, CFG.e_layers, CFG.n_heads = 128, 256, 2, 4
    CFG.epochs, CFG.batch_size = 2, 64
    CFG.embargo_safety_min = 240
    CFG.eval_max_windows = 40_000
    CFG.n_seeds_report = 1
elif CFG.profile == "tiny":
    # CPU-runnable smoke of the smoke: used to verify the notebook itself.
    CFG.grid_start, CFG.grid_end = "2021-01-02 00:00", "2021-03-31 23:59"
    CFG.train_end, CFG.val_end, CFG.test_end = (
        "2021-02-15 23:59", "2021-03-07 23:59", "2021-03-31 23:59")
    CFG.seq_len, CFG.pred_len = 120, 15
    CFG.train_stride = 240
    CFG.d_model, CFG.d_ff, CFG.e_layers, CFG.n_heads = 64, 128, 2, 4
    CFG.epochs, CFG.batch_size = 1, 32
    CFG.embargo_safety_min = 60
    CFG.horizons = (1, 5, 15)
    CFG.backtest_horizon = 15
    CFG.eval_max_windows = 8_000
    CFG.n_seeds_report = 1
    CFG.num_workers = 0
    CFG.fracdiff_width = 128
    CFG.run_vanilla_transformer = True
elif CFG.profile != "full":
    raise ValueError(f"unknown profile {CFG.profile!r}")

assert CFG.d_model % CFG.n_heads == 0, "d_model must be divisible by n_heads"

CFG.run_id = f"{CFG.profile}_L{CFG.seq_len}_H{CFG.pred_len}_d{CFG.d_model}_s{CFG.seed}"
RUN_DIR = WORK_DIR / "runs" / CFG.run_id
CKPT_DIR = WORK_DIR / "checkpoints" / CFG.run_id
for d in (RUN_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Checkpoints are READ from RESUME_DIR and always WRITTEN to CKPT_DIR, so a resumed
# session reads the previous session's read-only input dataset and writes its own output.
RESUME_DIR = Path(KAGGLE_RESUME_DIR) if KAGGLE_RESUME_DIR else CKPT_DIR

SESSION_T0 = time.time()


def hours_left() -> float:
    """Hours remaining in the self-imposed session budget."""
    return CFG.session_budget_hours - (time.time() - SESSION_T0) / 3600.0


set_seed(CFG.seed, CFG.deterministic)

print(f"profile      {CFG.profile}")
print(f"run_id       {CFG.run_id}")
print(f"grid         {CFG.grid_start} -> {CFG.grid_end}")
print(f"L={CFG.seq_len}  H={CFG.pred_len}  stride={CFG.train_stride}")
print(f"split gap    {CFG.split_gap_min:,} min  (purge {CFG.pred_len} + embargo {CFG.embargo_min:,})")
print(f"ckpt         {CKPT_DIR}")
print(f"resume from  {RESUME_DIR}"
      + ("   (same session)" if RESUME_DIR == CKPT_DIR else "   (previous session)"))
print(f"budget       {CFG.session_budget_hours:.1f} h with {CFG.reserve_hours:.1f} h reserved  (Kaggle hard cap: 12 h)")

<div style="background: linear-gradient(90deg, #1a1a2e, #16213e); border-left: 4px solid #e94560; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #f5a623; margin: 0 0 10px 0;">📅 The publication-lag table</h2>
  <p style="color: #ffd6a5; margin: 0 0 8px 0;"><strong>This is where leakage usually enters a macro-conditioned model.</strong> A row dated <code>2018-01-31</code> for CPI was <em>not knowable</em> on 31 January — CPI for January is published around 13 February. Joining it from its nominal date hands the model two weeks of the future, every month, for eight years.</p>  <p style="color: #ffd6a5; margin: 0 0 8px 0;">Each indicator below is shifted forward by its release lag <em>before</em> any join happens. Where a release date is uncertain the lag is rounded <strong>up</strong> to a whole extra period: losing a little signal is vastly cheaper than publishing a leaked result.</p>  <ul style="color: #ffd6a5; margin: 0; padding-left: 20px;">
    <li>Market-observed rates (Treasury yields, effective fed funds) sit inside a <em>monthly</em> file, so they are monthly <em>aggregates</em> — they get the same one-month lag as everything else, not daily treatment.</li>
    <li><code>GDP</code>/<code>GDP_Real</code> are quarterly values already forward-filled to monthly. They get a full quarter of lag, and are treated as a step function rather than differenced month-over-month.</li>
    <li>The daily USD index for date <em>d</em> is only knowable after <em>d</em>'s close, so it applies from <em>d</em> + 1 day.</li>
    <li><strong>Residual risk that no lag table can fix:</strong> these are <em>revised</em> values, not real-time ALFRED vintages. Even with perfect release lags, revisions leak a small amount of future information. This is stated again in the limitations section — it is a property of the dataset, not a bug in the pipeline.</li>
  </ul>
</div>

In [ ]:
# Release lag per macro column, in (months, days) added to the nominal month-end date.
# Conservative by construction: when in doubt, round up to a whole extra period.
RELEASE_LAG = {
    "CPI":                        (1, 13),   # ~13th of M+1, 13:30 UTC
    "CPI_Core":                   (1, 13),
    "PPI":                        (1, 14),
    "PCE":                        (2, 0),    # ~last business day of M+1 -> take 2 months
    "PCE_Core":                   (2, 0),
    "Non_Farm_Payroll":           (1, 7),    # 1st Friday of M+1
    "Unemployment_Rate":          (1, 7),
    "Labor_Force_Participation":  (1, 7),
    "Retail_Sales":               (1, 16),
    "Industrial_Production":      (1, 15),
    "Capacity_Utilization":       (1, 15),
    "GDP":                        (3, 0),    # quarterly; advance ~Q_end+30d -> take a full quarter
    "GDP_Real":                   (3, 0),
    "M1":                         (1, 0),
    "M2":                         (1, 0),
    "Bank_Credit":                (1, 0),
    "Consumer_Credit":            (2, 0),
    "Reserves":                   (1, 0),
    "Fed_Balance_Sheet":          (1, 0),
    "Fed_Funds_Rate":             (1, 0),    # monthly aggregate of a market-observed rate
    "Fed_Funds_Target":           (1, 0),
    "Fed_Funds_Target_Lower":     (1, 0),
    "Treasury_10Y":               (1, 0),
    "Treasury_2Y":                (1, 0),
    "Treasury_5Y":                (1, 0),
    "Real_Interest_Rate":         (1, 0),
    "Mortgage_Rate_30Y":          (1, 0),
    "Consumer_Sentiment":         (1, 0),
    "Inflation_Expectations_5Y":  (1, 0),
    "Inflation_Expectations_10Y": (1, 0),
    "Real_Broad_Dollar_Index":    (1, 0),    # dropped later - see the redundancy note in section 6
}

DXY_LAG_DAYS = 1          # daily index for date d is knowable from d+1 00:00 UTC

# Scheduled high-impact release times, used only for temporal flags (never for values).
RELEASE_MINUTES_UTC = {"cpi_nfp": (13, 30), "fomc": (19, 0)}

print(f"{len(RELEASE_LAG)} macro columns carry an explicit release lag")
print(f"longest lag: {max(RELEASE_LAG.values())} (months, days)")

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 4px solid #f48c06; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #ffd60a; margin: 0 0 10px 0;">📡 §3 · Loading &amp; Schema Contracts</h2>
  <p style="color: #ffb703; margin: 0 0 8px 0;"><strong>Every column in every parquet file is a <code>String</code>.</strong> The upstream CSV→parquet conversion wrote <code>dtype=str</code>, and missing values became <strong>empty strings</strong>, not nulls. A loader that ignores this either crashes or — far worse — silently coerces bad cells to null.</p>  <ul style="color: #ffb703; margin: 0; padding-left: 20px;">
    <li>Map <code>""</code> → <code>null</code> first, then cast with <code>strict=True</code> so a malformed cell <em>raises</em> instead of quietly vanishing.</li>
    <li>Assert the expected column set on load. Schema drift after a data refresh should fail loudly here, not produce a subtly wrong model three hours later.</li>
    <li>Normalise every timestamp to <strong>tz-aware UTC</strong>. BTC's <code>datetime</code> string is cross-checked against its epoch-millisecond <code>timestamp</code> column — they must agree exactly.</li>
    <li><code>data/raw/</code> is immutable. Nothing in this notebook writes to it.</li>
  </ul>
</div>

In [ ]:
def _f64(col: str, alias: str | None = None) -> pl.Expr:
    """String column -> Float64, with empty string treated as null and strict casting."""
    return pl.col(col).replace("", None).cast(pl.Float64, strict=True).alias(alias or col)


def _assert_schema(lf: pl.LazyFrame, expected: set[str], name: str) -> None:
    got = set(lf.collect_schema().names())
    missing, extra = expected - got, got - expected
    assert not missing, f"[{name}] missing columns: {sorted(missing)}"
    if extra:
        print(f"  [{name}] note: {len(extra)} unexpected column(s) ignored: {sorted(extra)[:5]}")


def load_btc(raw_dir: Path) -> pl.DataFrame:
    """BTC/USDT 1-minute OHLCV. Defines the master grid."""
    lf = pl.scan_parquet(str(raw_dir / "btc_usdt_binance_*.parquet"))
    _assert_schema(lf, {"timestamp", "open", "high", "low", "close", "volume", "datetime"}, "btc")
    df = (
        lf.select(
            pl.col("datetime")
              .str.strptime(pl.Datetime("us"), "%Y-%m-%d %H:%M:%S%:z")
              .dt.convert_time_zone("UTC")
              .alias("t"),
            pl.col("timestamp").cast(pl.Int64, strict=True).alias("_ts_ms"),
            _f64("open", "btc_open"), _f64("high", "btc_high"),
            _f64("low", "btc_low"), _f64("close", "btc_close"),
            _f64("volume", "btc_volume"),
        )
        .sort("t")
        .collect()
    )
    # The two time representations must agree exactly, or one of them is wrong.
    lhs = df["t"].dt.epoch(time_unit="ms")
    assert (lhs == df["_ts_ms"]).all(), "btc: datetime string disagrees with epoch-ms timestamp"
    return df.drop("_ts_ms")


def load_gold(raw_dir: Path, offset_hours: int) -> pl.DataFrame:
    """XAU/USD 1-minute OHLCV on the broker clock, shifted to UTC by `offset_hours`."""
    lf = pl.scan_parquet(str(raw_dir / "xauusd_2018_2026.parquet"))
    _assert_schema(lf, {"Date", "Timestamp", "Open", "High", "Low", "Close", "Volume"}, "gold")
    return (
        lf.select(
            (pl.concat_str([pl.col("Date"), pl.lit(" "), pl.col("Timestamp")])
               .str.strptime(pl.Datetime("us"), "%Y%m%d %H:%M:%S")
             - pl.duration(hours=offset_hours))
            .dt.replace_time_zone("UTC")
            .alias("t"),
            _f64("Open", "gold_open"), _f64("High", "gold_high"),
            _f64("Low", "gold_low"), _f64("Close", "gold_close"),
            _f64("Volume", "gold_volume"),
        )
        .sort("t")
        .collect()
    )


def load_dxy(raw_dir: Path) -> pl.DataFrame:
    """Daily broad trade-weighted USD index. Weekends are absent rows; holidays are empty strings."""
    lf = pl.scan_parquet(str(raw_dir / "US_Dollar_Index_2018_2026.parquet"))
    _assert_schema(lf, {"Date", "US_Dollar_Index"}, "dxy")
    return (
        lf.select(
            pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d").alias("date"),
            _f64("US_Dollar_Index", "dxy"),
        )
        .drop_nulls("dxy")          # holidays carry no observation at all
        .sort("date")
        .collect()
    )


def load_macro(raw_dir: Path) -> pl.DataFrame:
    """Monthly US macro block, month-end dated. Values are revised, not real-time vintages."""
    lf = pl.scan_parquet(str(raw_dir / "fed_economic_data_2018_2026.parquet"))
    cols = [c for c in lf.collect_schema().names() if c != "Date"]
    assert set(cols) >= set(RELEASE_LAG), f"macro: lag table references unknown columns"
    return (
        lf.select(
            pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d").alias("date"),
            *[_f64(c) for c in cols],
        )
        .sort("date")
        .collect()
    )


t0 = time.time()
btc_raw = load_btc(RAW_DIR)
dxy_raw = load_dxy(RAW_DIR)
macro_raw = load_macro(RAW_DIR)
print(f"btc    {btc_raw.height:>9,} rows   {btc_raw['t'][0]} -> {btc_raw['t'][-1]}")
print(f"dxy    {dxy_raw.height:>9,} rows   {dxy_raw['date'][0]} -> {dxy_raw['date'][-1]}")
print(f"macro  {macro_raw.height:>9,} rows   {macro_raw['date'][0]} -> {macro_raw['date'][-1]}")
print(f"\nloaded in {time.time() - t0:.1f}s")

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 4px solid #f48c06; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #ffd60a; margin: 0 0 10px 0;">🕵️ §4 · Validation Battery</h2>
  <p style="color: #ffb703; margin: 0 0 8px 0;">Nothing downstream is trustworthy if the inputs are not what the specification claims. This cell re-measures the facts rather than assuming them, and prints a report you should actually read.</p>  <ul style="color: #ffb703; margin: 0; padding-left: 20px;">
    <li><strong>Monotonicity &amp; duplicates</strong> — a duplicated minute silently doubles a bar's weight in every rolling window that contains it.</li>
    <li><strong>OHLC sanity</strong> — <code>low ≤ min(open, close) ≤ max(open, close) ≤ high</code>, all strictly positive.</li>
    <li><strong>Gap census</strong> — BTC has 31 known discontinuities (Binance maintenance), the largest ≈ 10 h. These are <em>flagged</em>, never interpolated: windows spanning them are dropped later.</li>
    <li><strong>Extreme-return census</strong> — every <code>|log return| &gt; 10%</code> in one minute is listed, not deleted. Those minutes are the phenomenon of interest (2020-03-12, 2021-05-19, FTX 2022-11), and removing them is how you build a model that fails exactly when it matters.</li>
  </ul>
</div>

In [ ]:
def validate_ohlcv(df: pl.DataFrame, prefix: str, name: str) -> dict:
    """Print a validation report for one OHLCV source and return its key statistics."""
    o, h, l, c = (f"{prefix}_{x}" for x in ("open", "high", "low", "close"))
    v = f"{prefix}_volume"
    n = df.height
    print(f"--- {name} " + "-" * (58 - len(name)))
    print(f"  rows              {n:,}")

    t = df["t"]
    assert t.is_sorted(), f"{name}: timestamps not sorted"
    n_dup = n - t.n_unique()
    print(f"  duplicate stamps  {n_dup:,}" + ("   <-- MUST BE ZERO" if n_dup else ""))
    assert n_dup == 0, f"{name}: {n_dup} duplicate timestamps"

    bad_ohlc = df.filter(
        (pl.col(l) > pl.min_horizontal(o, c)) | (pl.col(h) < pl.max_horizontal(o, c))
        | (pl.col(l) > pl.col(h)) | (pl.col(c) <= 0)
    ).height
    print(f"  OHLC violations   {bad_ohlc:,}" + ("   <-- inspect" if bad_ohlc else ""))
    if v in df.columns:
        print(f"  negative volume   {df.filter(pl.col(v) < 0).height:,}")

    nulls = {c_: df[c_].null_count() for c_ in df.columns if df[c_].null_count()}
    print(f"  nulls             {nulls if nulls else 'none'}")

    # gap census
    step = t.diff().dt.total_minutes().drop_nulls()
    gaps = step.filter(step > 1)
    print(f"  1-min steps       {(step == 1).sum():,}")
    print(f"  discontinuities   {gaps.len():,}")
    if gaps.len():
        big = gaps.sort(descending=True).head(5).to_list()
        print(f"  largest gaps      {[f'{g / 60:.2f}h' for g in big]}")
        buckets = [(1, 5), (5, 60), (60, 24 * 60), (24 * 60, 10**9)]
        hist = {f"{a}-{b}m": int(((gaps > a) & (gaps <= b)).sum()) for a, b in buckets}
        print(f"  gap histogram     {hist}")

    # extreme returns, on the source's own clock
    r = (pl.Series(np.log(df[c].to_numpy())).diff()).drop_nulls()
    ext = int((r.abs() > 0.10).sum())
    print(f"  |1-min logret|>10%  {ext:,}")
    if ext:
        idx = np.where(np.abs(r.to_numpy()) > 0.10)[0] + 1
        for i in idx[:6]:
            print(f"      {df['t'][int(i)]}  r={r[int(i) - 1]:+.4f}  close={df[c][int(i)]:,.2f}")
        if len(idx) > 6:
            print(f"      ... and {len(idx) - 6} more")
    print()
    return {"rows": n, "gaps": gaps.len(), "extremes": ext, "bad_ohlc": bad_ohlc}


rep_btc = validate_ohlcv(btc_raw, "btc", "BTC/USDT 1-min")

print("--- USD index (daily) " + "-" * 41)
print(f"  rows (non-null)   {dxy_raw.height:,}")
print(f"  range             {dxy_raw['date'][0]} -> {dxy_raw['date'][-1]}")
print(f"  level             {dxy_raw['dxy'].min():.2f} .. {dxy_raw['dxy'].max():.2f}")
_gapd = dxy_raw["date"].diff().dt.total_days().drop_nulls()
print(f"  calendar gaps     {dict(zip(*np.unique(_gapd.to_numpy(), return_counts=True)))}")

print("\n--- Macro (monthly) " + "-" * 43)
print(f"  rows              {macro_raw.height:,}   {macro_raw['date'][0]} -> {macro_raw['date'][-1]}")
_mn = {c: macro_raw[c].null_count() for c in macro_raw.columns if macro_raw[c].null_count()}
print(f"  nulls             {_mn if _mn else 'none'}")
_gdp = macro_raw["GDP"].drop_nulls()
print(f"  GDP distinct/rows {_gdp.n_unique()}/{_gdp.len()}   (quarterly, forward-filled to monthly)")

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 4px solid #f48c06; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #ffd60a; margin: 0 0 10px 0;">🌍 §5 · Resolving the Gold Timezone — Empirically</h2>
  <p style="color: #ffb703; margin: 0 0 8px 0;">The gold file's timestamps carry <strong>no timezone label</strong>. Its first bar is <code>2018-01-01 23:00</code>, and its weekly closure pattern is consistent with a broker server clock — commonly UTC+2/UTC+3 — rather than UTC. <strong>Guessing wrong injects a fixed look-ahead or look-behind bias into every cross-asset feature</strong>, so this is measured, not assumed.</p>  <p style="color: #ffb703; margin: 0 0 8px 0;">Two independent lines of evidence are computed below. They agree.</p>  <ul style="color: #ffb703; margin: 0; padding-left: 20px;">
    <li><strong>Structural.</strong> Where does the weekly closure fall? A gold market anchored to New York closes at 17:00 New York time — that is 22:00 UTC in winter and 21:00 UTC in summer.</li>
    <li><strong>Statistical.</strong> Shift the gold series by each candidate offset from −12 h to +12 h and measure the contemporaneous correlation of 1-minute BTC and gold log returns. The true offset should stand out sharply; every wrong offset should look like noise.</li>
  </ul>  <p style="color: #ffb703; margin: 0 0 8px 0;"><strong>Result (pre-computed on the full dataset, reproduced by the cell below):</strong> the last bar before the weekly break is <code>Fri 21:59</code> in winter and <code>Fri 20:59</code> in summer, reopening <code>Sun 23:00</code> / <code>Sun 22:00</code> — exactly New York 17:00 close and 18:00 reopen, in both DST regimes. The correlation scan peaks at <code>+0h</code> with <code>ρ = +0.0626</code> while every other offset sits at <code>±0.002</code>, a 30× separation. <strong>The gold file is already in UTC; the offset is 0.</strong> CLAUDE.md §17 verification task 1 is closed.</p>
</div>

In [ ]:
def detect_gold_offset(raw_dir: Path, scan_years: tuple = (2021, 2024)) -> dict:
    """Recover the gold broker-clock UTC offset from structure + BTC cross-correlation.

    Returns the argmax offset in whole hours along with the full evidence table.
    """
    g = load_gold(raw_dir, offset_hours=0).select("t", "gold_close")
    t = g["t"].dt.replace_time_zone(None).to_numpy()
    step = np.diff(t).astype("timedelta64[m]").astype(np.int64)

    def _season(arr):
        m = np.array([int(str(x)[5:7]) for x in arr])
        return np.isin(m, [11, 12, 1, 2]), np.isin(m, [4, 5, 6, 7, 8, 9])

    def _wd(x):   # 1970-01-01 was a Thursday
        return ["Thu", "Fri", "Sat", "Sun", "Mon", "Tue", "Wed"][int(np.datetime64(x, "D").astype(int)) % 7]

    wk = np.where(step >= 24 * 60)[0]
    close_b, open_b = t[wk], t[wk + 1]
    win, summ = _season(close_b)
    struct = {}
    for label, sel in (("winter", win), ("summer", summ)):
        cc = np.unique([f"{_wd(x)} {str(x)[11:16]}" for x in close_b[sel]], return_counts=True)
        oo = np.unique([f"{_wd(x)} {str(x)[11:16]}" for x in open_b[sel]], return_counts=True)
        struct[label] = {"close": cc[0][cc[1].argmax()], "open": oo[0][oo[1].argmax()]}
        print(f"  weekly break ({label}):  close {struct[label]['close']}  ->  reopen {struct[label]['open']}")

    lo, hi = datetime(scan_years[0], 1, 1, tzinfo=timezone.utc), datetime(scan_years[1], 1, 1, tzinfo=timezone.utc)
    b = (btc_raw.filter(pl.col("t").is_between(lo, hi))
                .select("t", pl.col("btc_close").log().diff().alias("br")).drop_nulls())
    gr = g.select("t", pl.col("gold_close").log().diff().alias("gr")).drop_nulls()

    scan = []
    for off in range(-12, 13):
        j = b.join(gr.with_columns(pl.col("t") - pl.duration(hours=off)), on="t", how="inner")
        if j.height < 10_000:
            continue
        x, y = j["br"].to_numpy(), j["gr"].to_numpy()
        m = np.isfinite(x) & np.isfinite(y) & (y != 0.0)
        scan.append((off, float(np.corrcoef(x[m], y[m])[0, 1]), int(m.sum())))

    best = max(scan, key=lambda r: abs(r[1]))
    runner = max((s for s in scan if s[0] != best[0]), key=lambda r: abs(r[1]))
    print(f"\n  correlation argmax:  offset {best[0]:+d}h   rho={best[1]:+.5f}  (n={best[2]:,})")
    print(f"  next best         :  offset {runner[0]:+d}h   rho={runner[1]:+.5f}"
          f"   -> separation {abs(best[1] / runner[1]):.0f}x")
    return {"offset": best[0], "scan": scan, "structure": struct}


if CFG.gold_utc_offset_h == "auto":
    _det = detect_gold_offset(RAW_DIR)
    GOLD_OFFSET_H = _det["offset"]
    fig, ax = plt.subplots(figsize=(9, 3.2))
    offs = [s[0] for s in _det["scan"]]
    cors = [abs(s[1]) for s in _det["scan"]]
    ax.bar(offs, cors, color=[CAT[1] if o == GOLD_OFFSET_H else GRID for o in offs], width=0.7)
    ax.annotate(f"offset {GOLD_OFFSET_H:+d}h", (GOLD_OFFSET_H, max(cors)),
                textcoords="offset points", xytext=(0, 6), ha="center", color=CAT[1], fontsize=9)
    finish(ax, "Gold-clock offset scan  |  contemporaneous BTC-gold 1-min return correlation",
           "candidate offset (hours)", "|rho|")
    plt.tight_layout(); plt.show()
else:
    GOLD_OFFSET_H = int(CFG.gold_utc_offset_h)
    print(f"  using pinned offset {GOLD_OFFSET_H:+d}h "
          f"(set CFG.gold_utc_offset_h='auto' to re-derive it from the data)")

gold_raw = load_gold(RAW_DIR, GOLD_OFFSET_H)
print(f"\ngold   {gold_raw.height:>9,} rows   {gold_raw['t'][0]} -> {gold_raw['t'][-1]}")
rep_gold = validate_ohlcv(gold_raw, "gold", "XAU/USD 1-min (UTC-normalised)")

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 4px solid #48cae4; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🧬 §6 · Multi-Granularity Alignment</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;">Four series sampled at 1 min / 1 min-with-closures / 1 day / 1 month must become <strong>one matrix on the BTC minute grid</strong> without leaking the future. This section is the part that most determines whether the project succeeds.</p>  <p style="color: #ade8f4; margin: 0 0 8px 0;"><strong>The governing principle:</strong> at master timestamp <em>t</em>, a feature may use only information a real observer would have possessed at or before <em>t</em>. Every join is therefore a <strong>backward as-of join</strong> — never nearest, never forward.</p>  <p style="color: #ade8f4; margin: 0 0 8px 0;"><strong>The architectural decision that makes this tractable:</strong> each source's features are computed <em>on its own native clock first</em>, and only the finished features are as-of joined onto the minute grid. Doing it the other way round — forward-filling gold prices onto every minute and then differencing — manufactures long runs of exactly-zero returns that corrupt every volatility and correlation estimate downstream. It also keeps the joined frame at ~30 columns instead of ~45, which matters at 4.4M rows.</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li><strong>BTC gaps</strong> are materialised as explicit rows carrying <code>btc_is_synthetic=1</code>, price forward-filled and volume zero. They are never interpolated, and windows overlapping them are dropped at sampling time.</li>
    <li><strong>Gold staleness</strong> is a feature, not a nuisance: <code>log1p(minutes since last gold bar)</code> plus a binary <code>gold_market_open</code>. The model is told how stale its inputs are.</li>
    <li><strong>The weekend gap return</strong> (Friday close → Sunday open) is a genuine two-day return. It is not spread across the weekend minutes; it appears at the reopening bar and decays with a 60-minute time constant.</li>
    <li><strong>Macro and USD index</strong> are shifted by the publication-lag table <em>before</em> joining, and each carries an age feature (<code>macro_age_days</code>, <code>dxy_age_days</code>).</li>
  </ul>
</div>

In [ ]:
UTC = timezone.utc


def ts(s: str) -> datetime:
    """'YYYY-MM-DD HH:MM' -> tz-aware UTC datetime."""
    return datetime.strptime(s, "%Y-%m-%d %H:%M").replace(tzinfo=UTC)


In [ ]:
# Warm-up prefix: the longest trailing window (1440) plus one lookback window,
# so that the first *usable* window at grid_start already has complete features.
WARMUP_MIN = 1440 + CFG.seq_len + 60
GRID_LO = max(ts(CFG.grid_start) - timedelta(minutes=WARMUP_MIN), btc_raw["t"][0])
GRID_HI = ts(CFG.grid_end)

grid = pl.datetime_range(GRID_LO, GRID_HI, interval="1m", time_zone="UTC", eager=True).alias("t")
master = pl.DataFrame({"t": grid}).join(
    btc_raw.filter(pl.col("t").is_between(GRID_LO, GRID_HI)), on="t", how="left"
)
n_synth = master["btc_close"].null_count()
master = master.with_columns(
    pl.col("btc_close").is_null().cast(pl.Float32).alias("btc_is_synthetic")
).with_columns(
    # forward-fill price through a gap (never backward), zero the volume
    *[pl.col(c).forward_fill() for c in ("btc_open", "btc_high", "btc_low", "btc_close")],
    pl.col("btc_volume").fill_null(0.0),
)
assert master["btc_close"].null_count() == 0, "grid starts before the first BTC bar"

print(f"master grid   {master.height:,} rows   {GRID_LO} -> {GRID_HI}")
print(f"  warm-up prefix   {WARMUP_MIN:,} min before {CFG.grid_start}")
print(f"  synthetic bars   {n_synth:,}  ({100 * n_synth / master.height:.4f}% of the grid)")

In [ ]:
# ---------------------------------------------------------------------------
# Gold features, computed on GOLD'S OWN CLOCK, then as-of joined.
# ---------------------------------------------------------------------------
GAP_MIN = 120          # a break longer than this is a closure, and its return is a "gap return"

gold_f = (
    gold_raw.with_columns(pl.col("gold_close").log().alias("_lg"))
    .with_columns(
        pl.col("t").diff().dt.total_minutes().alias("_step"),
        pl.col("_lg").diff().alias("gold_logret_1"),
    )
    .with_columns(
        # a return realised across a closure is a gap return, not a 1-minute return
        pl.when(pl.col("_step") > GAP_MIN).then(pl.col("gold_logret_1"))
          .otherwise(None).alias("_gap_ret"),
    )
    .with_columns(
        pl.when(pl.col("_step") > GAP_MIN).then(0.0)
          .otherwise(pl.col("gold_logret_1")).alias("gold_logret_1"),
        pl.when(pl.col("_step") > GAP_MIN).then(pl.col("t"))
          .otherwise(None).alias("_reopen_t"),
    )
    .with_columns(
        pl.col("_gap_ret").forward_fill().fill_null(0.0).alias("gold_gap_ret"),
        pl.col("_reopen_t").forward_fill().alias("gold_reopen_t"),
        (pl.col("_lg") - pl.col("_lg").shift(5)).alias("gold_logret_5"),
        (pl.col("_lg") - pl.col("_lg").shift(60)).alias("gold_logret_60"),
        (pl.col("_lg") - pl.col("_lg").shift(1440)).alias("gold_logret_1440"),
    )
    .with_columns(
        pl.col("gold_logret_1").pow(2).rolling_sum(60).sqrt().alias("gold_rv_60"),
        pl.col("t").alias("_gold_obs_t"),
    )
    .select("t", "_gold_obs_t", "gold_logret_1", "gold_logret_5", "gold_logret_60",
            "gold_logret_1440", "gold_rv_60", "gold_gap_ret", "gold_reopen_t")
)
print(f"gold features on gold clock: {gold_f.height:,} rows, {gold_f.width - 1} columns")

# ---------------------------------------------------------------------------
# Cross-asset features, also on gold's clock: BTC is as-of joined ONTO gold so
# that correlations are never contaminated by forward-filled zero gold returns.
# ---------------------------------------------------------------------------
xa = (
    gold_raw.select("t", pl.col("gold_close").log().alias("lg"))
    .join_asof(btc_raw.select("t", pl.col("btc_close").log().alias("lb")),
               on="t", strategy="backward")
    .drop_nulls()
    .with_columns(pl.col("lg").diff().alias("rg"), pl.col("lb").diff().alias("rb"))
    .drop_nulls()
)
LL = 5   # lead-lag probe, in gold-clock minutes
xa = (
    xa.with_columns(
        pl.rolling_corr(pl.col("rb"), pl.col("rg"), window_size=60).alias("xa_corr_60"),
        pl.rolling_corr(pl.col("rb"), pl.col("rg"), window_size=1440).alias("xa_corr_1440"),
        (pl.rolling_cov(pl.col("rb"), pl.col("rg"), window_size=1440)
         / (pl.col("rg").rolling_var(1440) + 1e-12)).alias("xa_beta_1440"),
        pl.rolling_corr(pl.col("rb"), pl.col("rg").shift(LL), window_size=240)
          .alias("xa_gold_leads_btc"),
        pl.rolling_corr(pl.col("rg"), pl.col("rb").shift(LL), window_size=240)
          .alias("xa_btc_leads_gold"),
    )
    .with_columns((pl.col("lb") - pl.col("xa_beta_1440") * pl.col("lg")).alias("_spread"))
    .with_columns(
        ((pl.col("_spread") - pl.col("_spread").rolling_mean(1440))
         / (pl.col("_spread").rolling_std(1440) + 1e-12)).alias("xa_spread_z")
    )
    .select("t", "xa_corr_60", "xa_corr_1440", "xa_beta_1440",
            "xa_gold_leads_btc", "xa_btc_leads_gold", "xa_spread_z")
)
print(f"cross-asset features on gold clock: {xa.height:,} rows, {xa.width - 1} columns")

In [ ]:
# ---------------------------------------------------------------------------
# USD index: daily clock. Available from d+1 00:00 UTC (after d's close).
# ---------------------------------------------------------------------------
def _pct_rank_252(x: np.ndarray) -> np.ndarray:
    """Trailing percentile rank of the latest value within a 252-observation window."""
    out = np.full(x.shape, np.nan)
    for i in range(251, len(x)):
        w = x[i - 251: i + 1]
        out[i] = (w < w[-1]).mean()
    return out


_d = dxy_raw.with_columns(pl.col("dxy").log().alias("_l"))
dxy_f = _d.with_columns(
    (pl.col("_l") - pl.col("_l").shift(1)).alias("dxy_logret_1d"),
    (pl.col("_l") - pl.col("_l").shift(5)).alias("dxy_logret_5d"),
    (pl.col("_l") - pl.col("_l").shift(21)).alias("dxy_logret_21d"),
    pl.Series("dxy_pctrank_252", _pct_rank_252(_d["dxy"].to_numpy())),
).with_columns(
    # `t` is when the value becomes KNOWABLE; `_dxy_obs_t` is the date it describes.
    # Joining on `t` enforces the lag; measuring age from `_dxy_obs_t` makes the
    # staleness feature report how old the underlying observation actually is.
    (pl.col("date").cast(pl.Datetime("us")) + pl.duration(days=DXY_LAG_DAYS))
    .dt.replace_time_zone("UTC").alias("t"),
    pl.col("date").cast(pl.Datetime("us")).dt.replace_time_zone("UTC").alias("_dxy_obs_t"),
).select("t", "dxy_logret_1d", "dxy_logret_5d", "dxy_logret_21d", "dxy_pctrank_252",
         "_dxy_obs_t")
print(f"dxy features (release-lagged +{DXY_LAG_DAYS}d): {dxy_f.height:,} rows")

# ---------------------------------------------------------------------------
# Macro: monthly clock. Each column shifted by its own release lag, then the
# whole block compressed to a handful of variates.
#
# `Real_Broad_Dollar_Index` is DROPPED. Measured against the daily USD index
# file it has level correlation 0.973 and month-over-month log-change
# correlation 0.989 - it is a near-duplicate variate that would inflate
# cross-variate attention on a redundant signal. The daily file is strictly
# more informative, so the monthly column goes.
# ---------------------------------------------------------------------------
MACRO_DROP = {"Real_Broad_Dollar_Index"}
MACRO_USE = [c for c in macro_raw.columns if c != "date" and c not in MACRO_DROP]

# Availability timestamp per column group. +1 extra day on top of the tabled lag,
# because a release published at 13:30 UTC is not knowable at 00:00 UTC that day.
avail = {}
for col in MACRO_USE:
    m, d = RELEASE_LAG[col]
    avail[col] = f"{m}mo{d + 1}d"

STEP_COLS = {"GDP", "GDP_Real"}      # quarterly, forward-filled: differencing them month-over-month is meaningless

mac = macro_raw.sort("date")
exprs = []
for c in MACRO_USE:
    s = pl.col(c)
    exprs.append((s / s.shift(12) - 1.0).alias(f"{c}__yoy"))
    if c not in STEP_COLS:
        exprs.append((s / s.shift(1) - 1.0).alias(f"{c}__mom"))
    exprs.append(((s - s.rolling_mean(36, min_samples=18))
                  / (s.rolling_std(36, min_samples=18) + 1e-12)).alias(f"{c}__z36"))
mac = mac.with_columns(exprs)

# Explicit, economically-motivated regime features kept out of the PCA block.
mac = mac.with_columns(
    (pl.col("Treasury_10Y") - pl.col("Treasury_2Y")).alias("macro_yield_curve"),
    (pl.col("Fed_Funds_Rate") - 100 * pl.col("CPI__yoy")).alias("macro_real_rate"),
    (pl.col("M2").log().diff(12)).alias("macro_m2_impulse"),
    (pl.col("Fed_Balance_Sheet").log().diff(12)).alias("macro_bs_impulse"),
)
print(f"macro: {len(MACRO_USE)} indicators -> {mac.width - 1} derived columns "
      f"(dropped {sorted(MACRO_DROP)})")

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 4px solid #48cae4; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🧮 Compressing the macro block</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;">31 indicators × three transforms each is ~90 columns. Feeding that to a model whose attention runs <em>over variates</em> would drown the four minute-frequency signals that actually carry short-horizon information. Two levers control this, and both are fitted on the training split alone:</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li><strong>PCA to 4 components.</strong> Loadings are fitted only on months whose <em>availability</em> timestamp falls inside the training range — not months whose nominal date does. That distinction is the whole point of the lag table.</li>
    <li><strong>Four hand-picked regime features kept outside the PCA</strong> — yield-curve slope, real rate, M2 impulse, balance-sheet impulse — because they are economically interpretable and we want to read them off the attention map later.</li>
  </ul>
</div>

In [ ]:
# Availability timestamp: nominal month-end shifted by that column's release lag.
mac_av = mac.with_columns([
    (pl.col("date").cast(pl.Datetime("us")).dt.offset_by(avail[c]))
    .dt.replace_time_zone("UTC").alias(f"_av__{c}")
    for c in MACRO_USE
])

TRAIN_END_TS = ts(CFG.train_end)
Z_COLS = [c for c in mac.columns if "__" in c and not c.startswith("_av")]

# One as-of join per distinct lag group keeps this to a handful of joins.
groups: dict[str, list[str]] = {}
for c in MACRO_USE:
    groups.setdefault(avail[c], []).append(c)

blocks = []
for lag_str, cols in groups.items():
    sub = [z for z in Z_COLS if z.split("__")[0] in cols]
    if not sub:
        continue
    frame = (mac_av.select(pl.col(f"_av__{cols[0]}").alias("t"), *sub)
                   .drop_nulls("t").sort("t"))
    blocks.append((lag_str, frame))
print(f"{len(blocks)} distinct release-lag groups -> {len(blocks)} as-of joins")

# PCA on the standardised macro transform matrix, fitted on TRAIN months only.
pca_frame = blocks[0][1].select("t")
for _, fr in blocks:
    pca_frame = pca_frame.join(fr, on="t", how="full", coalesce=True)
pca_frame = pca_frame.sort("t").fill_null(strategy="forward").drop_nulls()

feat_cols = [c for c in pca_frame.columns if c != "t"]
M = pca_frame.select(feat_cols).to_numpy().astype(np.float64)
train_mask = (pca_frame["t"] <= TRAIN_END_TS).to_numpy()
assert train_mask.sum() >= 12, "not enough in-train macro months to fit PCA"

# Ratio transforms (x/x.shift - 1) go infinite wherever the lagged value is zero,
# and 0/0 gives NaN. Left in place these make the SVD non-convergent, so they are
# neutralised to the column mean *before* the loadings are fitted.
n_bad_macro = int((~np.isfinite(M)).sum())
M[~np.isfinite(M)] = np.nan
col_ok = np.isfinite(M[train_mask]).sum(0) >= max(6, int(0.5 * train_mask.sum()))
col_ok &= np.nanstd(M[train_mask], axis=0) > 1e-10
M, feat_cols = M[:, col_ok], [c for c, k in zip(feat_cols, col_ok) if k]
mu_m = np.nanmean(M[train_mask], axis=0)
M = np.where(np.isfinite(M), M, mu_m)
print(f"macro matrix: {n_bad_macro:,} non-finite cells neutralised, "
      f"{int((~col_ok).sum())} degenerate columns dropped -> {M.shape[1]} inputs")

sd_m = np.std(M[train_mask], axis=0) + 1e-12
Zt = np.clip((M[train_mask] - mu_m) / sd_m, -10, 10)
try:
    U, S, Vt = np.linalg.svd(Zt, full_matrices=False)
except np.linalg.LinAlgError:                       # fall back to the covariance eigendecomposition
    ev, V = np.linalg.eigh(Zt.T @ Zt)
    order = np.argsort(ev)[::-1]
    S, Vt = np.sqrt(np.clip(ev[order], 0, None)), V[:, order].T
K = min(CFG.macro_n_pca, Vt.shape[0], int(train_mask.sum()) - 1)
W_pca = Vt[:K].T                                   # (n_features, K)
evr = (S**2 / (S**2).sum())[:K]
Zall = np.clip((M - mu_m) / sd_m, -10, 10)
PC = Zall @ W_pca
PC /= (PC[train_mask].std(0) + 1e-12)              # unit variance on the train split

macro_f = pl.DataFrame({"t": pca_frame["t"]}).with_columns(
    *[pl.Series(f"macro_pc{i + 1}", PC[:, i]) for i in range(K)]
).join(
    mac_av.select(
        pl.col("_av__M2").alias("t"),
        "macro_yield_curve", "macro_real_rate", "macro_m2_impulse", "macro_bs_impulse",
    ).drop_nulls("t"),
    on="t", how="left", coalesce=True,
).sort("t").fill_null(strategy="forward").with_columns(pl.col("t").alias("_macro_obs_t"))

# One row per distinct availability timestamp across all lag groups, not per calendar
# month: at each timestamp exactly one group carries fresh data and the rest are
# forward-filled, which is precisely the information set a real observer would hold.
print(f"macro PCA fitted on {int(train_mask.sum())} in-train availability timestamps "
      f"(of {len(M)} total), {len(feat_cols)} inputs -> {K} components")
print(f"  explained variance ratio: {np.round(evr, 4).tolist()}  (cumulative {evr.sum():.3f})")
print(f"macro features joined to the grid: {macro_f.width - 2} + age")

In [ ]:
# ---------------------------------------------------------------------------
# The joins. All backward as-of, all on UTC. Nothing here may look forward.
# ---------------------------------------------------------------------------
t0 = time.time()
master = (
    master
    .join_asof(gold_f.sort("t"), on="t", strategy="backward")
    .join_asof(xa.sort("t"), on="t", strategy="backward")
    .join_asof(dxy_f.sort("t"), on="t", strategy="backward")
    .join_asof(macro_f.sort("t"), on="t", strategy="backward")
)

master = master.with_columns(
    ((pl.col("t") - pl.col("_gold_obs_t")).dt.total_minutes()).alias("_gold_stale_min"),
    ((pl.col("t") - pl.col("_dxy_obs_t")).dt.total_days()).alias("dxy_age_days"),
    ((pl.col("t") - pl.col("_macro_obs_t")).dt.total_days()).alias("macro_age_days"),
    ((pl.col("t") - pl.col("gold_reopen_t")).dt.total_minutes()).alias("_since_reopen"),
).with_columns(
    pl.col("_gold_stale_min").log1p().alias("gold_staleness"),
    (pl.col("_gold_stale_min") <= 2).cast(pl.Float32).alias("gold_market_open"),
    # the weekend gap return appears at the reopening bar and decays over ~1 h
    (pl.col("gold_gap_ret") * (-pl.col("_since_reopen") / 60.0).exp()).alias("gold_gap_decay"),
)

# ---- alignment assertions ------------------------------------------------
assert master.height == grid.len(), "master row count != minute grid length"
assert master["t"].is_sorted() and master["t"].n_unique() == master.height
assert (master["_gold_stale_min"].drop_nulls() >= 0).all(), "gold as-of join looked forward"
assert (master["dxy_age_days"].drop_nulls() >= DXY_LAG_DAYS).all(), "dxy applied before its release lag"
assert (master["macro_age_days"].drop_nulls() >= 0).all(), "macro as-of join looked forward"

print(f"joined in {time.time() - t0:.1f}s   master shape {master.shape}")
print(f"  gold staleness   median {master['_gold_stale_min'].median():.0f} min, "
      f"p99 {master['_gold_stale_min'].quantile(0.99):.0f} min, "
      f"max {master['_gold_stale_min'].max():.0f} min")
print(f"  dxy age          median {master['dxy_age_days'].median():.1f} d, "
      f"max {master['dxy_age_days'].max():.0f} d")
print(f"  macro age        median {master['macro_age_days'].median():.0f} d, "
      f"max {master['macro_age_days'].max():.0f} d")
_first_ok = master.drop_nulls(["macro_pc1", "dxy_logret_1d", "gold_logret_1"])["t"][0]
print(f"  first row with every exogenous block available: {_first_ok}")

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 4px solid #48cae4; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🔬 §7 · Feature Engineering</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;">Every feature is <strong>causal</strong>: computed from a trailing window ending at <em>t</em> inclusive. Section 12 proves this with a shift test rather than asserting it.</p>  <p style="color: #ade8f4; margin: 0 0 8px 0;"><strong>Variate 0 is <code>btc_logret_1</code> — this is load-bearing, not cosmetic.</strong> iTransformer's instance normalisation denormalises its output using the statistics of the target variate, so the prediction lands in exactly the units of variate 0. Making variate 0 the one-minute log return means the model's output <em>is</em> the future one-minute log-return path, and the <em>h</em>-minute cumulative return is just its cumulative sum — no separate target tensor, no unit mismatch.</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li><strong>Target:</strong> <code>y = [r_{t+1}, …, r_{t+H}]</code>, one-minute log returns. Forecasting the <em>level</em> would produce a spectacular R² by echoing <code>close_t</code> while carrying zero information — any MSE reported on price levels in this domain is meaningless.</li>
    <li><strong>Fractional differentiation</strong> of <code>log(close)</code> with the smallest <em>d</em> that passes ADF at 95% on the training split — it preserves the memory that integer differencing destroys (López de Prado, ch. 5).</li>
    <li><strong>Four range-based volatility estimators</strong> (Parkinson, Garman–Klass, Rogers–Satchell, Yang–Zhang) plus realised volatility, bipower variation, and the <code>RV − BV</code> jump component.</li>
    <li><strong>Momentum indicators are deliberately capped at four.</strong> Hundreds of collinear indicators degrade cross-variate attention — the whole mechanism this architecture relies on.</li>
    <li><strong>Order-flow features are absent by design.</strong> The Binance export carries base-asset volume only — no quote volume, no trade count, no taker-buy split — so anything requiring them would be invented rather than measured.</li>
  </ul>
</div>

In [ ]:
EPS = 1e-12


def frac_weights(d: float, width: int, thresh: float = 1e-5) -> np.ndarray:
    """Fixed-width fractional-differentiation weights, longest-lag first."""
    w = [1.0]
    for k in range(1, width):
        wk = -w[-1] * (d - k + 1) / k
        if abs(wk) < thresh:
            break
        w.append(wk)
    return np.array(w[::-1])


def frac_diff(x: np.ndarray, d: float, width: int) -> np.ndarray:
    """Causal fixed-width frac-diff: output[i] uses only x[i-len(w)+1 .. i]."""
    w = frac_weights(d, width)
    out = np.full(x.shape, np.nan)
    if len(x) >= len(w):
        out[len(w) - 1:] = np.convolve(x, w, mode="valid")
    return out


def adf_stat(x: np.ndarray, lags: int = 8) -> float:
    """Augmented Dickey-Fuller t-statistic on the lagged-level coefficient (no p-value)."""
    x = x[np.isfinite(x)]
    dx = np.diff(x)
    n = len(dx) - lags
    if n < 100:
        return np.nan
    Y = dx[lags:]
    cols = [x[lags:-1], np.ones(n)] + [dx[lags - i: -i] for i in range(1, lags + 1)]
    X = np.column_stack(cols)
    beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
    resid = Y - X @ beta
    s2 = resid @ resid / (n - X.shape[1])
    se = np.sqrt(s2 * np.linalg.pinv(X.T @ X)[0, 0])
    return float(beta[0] / se)


ADF_CRIT_95 = -2.86     # Dickey-Fuller critical value, constant-only, large sample


def roll_moments(col: str, w: int) -> list[pl.Expr]:
    """Rolling skewness and excess kurtosis via raw moments (version-independent)."""
    x = pl.col(col)
    m1, m2, m3, m4 = (x.rolling_mean(w), x.pow(2).rolling_mean(w),
                      x.pow(3).rolling_mean(w), x.pow(4).rolling_mean(w))
    var = m2 - m1**2
    c3 = m3 - 3 * m1 * m2 + 2 * m1**3
    c4 = m4 - 4 * m1 * m3 + 6 * m1**2 * m2 - 3 * m1**4
    return [(c3 / (var.pow(1.5) + EPS)).alias(f"btc_skew_{w}"),
            (c4 / (var.pow(2) + EPS) - 3.0).alias(f"btc_kurt_{w}")]

In [ ]:
def build_features(m: pl.DataFrame, cfg: Config, frac_d: float) -> tuple[pl.DataFrame, dict]:
    """Build every feature block on the master minute grid.

    Pure function of `m` - this is what makes the shift test in section 12 possible.
    Returns the feature frame and a name->group mapping used by the ablation table.
    """
    groups: dict[str, str] = {}

    def reg(names, grp):
        for n in names:
            groups[n] = grp

    o, h, l, c, v = ("btc_open", "btc_high", "btc_low", "btc_close", "btc_volume")
    m = m.with_columns(
        pl.col(c).log().alias("_lc"),
        (pl.col(h) / pl.col(l)).log().alias("_hl"),
        (pl.col(c) / pl.col(o)).log().alias("_co"),
    )

    # ---- BTC price -------------------------------------------------------
    px = [pl.col("_lc").diff().alias("btc_logret_1")]
    for k in (5, 15, 30, 60, 240, 1440):
        px.append((pl.col("_lc") - pl.col("_lc").shift(k)).alias(f"btc_logret_{k}"))
    m = m.with_columns(px)
    reg([f"btc_logret_{k}" for k in (1, 5, 15, 30, 60, 240, 1440)], "btc_price")

    m = m.with_columns(
        pl.Series("btc_fracdiff", frac_diff(m["_lc"].to_numpy(), frac_d, cfg.fracdiff_width))
    )
    reg(["btc_fracdiff"], "btc_price")

    r = pl.col("btc_logret_1")
    vol = []
    for w in (15, 60, 1440):
        vol.append(r.pow(2).rolling_sum(w).sqrt().alias(f"btc_rv_{w}"))
    # bipower variation is robust to jumps; RV - BV isolates the jump component
    bpv = (r.abs() * r.abs().shift(1)).rolling_sum(60) * (math.pi / 2)
    vol += [
        bpv.sqrt().alias("btc_bpv_60"),
        (r.pow(2).rolling_sum(60) - bpv).clip(lower_bound=0.0).sqrt().alias("btc_jump_60"),
    ]
    for w in (60, 1440):
        vol.append((pl.col("_hl").pow(2).rolling_mean(w) / (4 * math.log(2))).sqrt()
                   .alias(f"btc_parkinson_{w}"))
    vol += [
        (0.5 * pl.col("_hl").pow(2).rolling_mean(60)
         - (2 * math.log(2) - 1) * pl.col("_co").pow(2).rolling_mean(60))
        .clip(lower_bound=0.0).sqrt().alias("btc_garman_klass_60"),
        (((pl.col(h) / pl.col(c)).log() * (pl.col(h) / pl.col(o)).log()
          + (pl.col(l) / pl.col(c)).log() * (pl.col(l) / pl.col(o)).log())
         .rolling_mean(60)).clip(lower_bound=0.0).sqrt().alias("btc_rogers_satchell_60"),
        r.abs().rolling_max(60).alias("btc_maxabs_60"),
        ((pl.col(c) - pl.col(l).rolling_min(60))
         / (pl.col(h).rolling_max(60) - pl.col(l).rolling_min(60) + EPS)).alias("btc_closepos_60"),
    ]
    m = m.with_columns(vol + roll_moments("btc_logret_1", 60))
    # Yang-Zhang combines overnight, open-to-close and Rogers-Satchell variance
    k_yz = 0.34 / (1.34 + (1441) / (1439))
    m = m.with_columns(
        ((pl.col("_lc") - pl.col("_lc").shift(1)).pow(2).rolling_mean(1440) * (1 - k_yz)
         + pl.col("_co").pow(2).rolling_mean(1440) * k_yz
         + pl.col("btc_rogers_satchell_60").pow(2).rolling_mean(1440))
        .clip(lower_bound=0.0).sqrt().alias("btc_yang_zhang_1440")
    )
    reg(["btc_rv_15", "btc_rv_60", "btc_rv_1440", "btc_bpv_60", "btc_jump_60",
         "btc_parkinson_60", "btc_parkinson_1440", "btc_garman_klass_60",
         "btc_rogers_satchell_60", "btc_yang_zhang_1440", "btc_skew_60", "btc_kurt_60",
         "btc_maxabs_60", "btc_closepos_60"], "btc_price")

    # ---- BTC volume / liquidity -----------------------------------------
    lv = pl.col(v).log1p()
    vwap = (pl.col(c) * pl.col(v)).rolling_sum(60) / (pl.col(v).rolling_sum(60) + EPS)
    # Corwin-Schultz two-bar high-low spread estimator
    beta_cs = pl.col("_hl").pow(2) + pl.col("_hl").pow(2).shift(1)
    gamma_cs = (pl.max_horizontal(pl.col(h), pl.col(h).shift(1))
                / pl.min_horizontal(pl.col(l), pl.col(l).shift(1))).log().pow(2)
    den = 3 - 2 * math.sqrt(2)
    alpha_cs = ((2 * beta_cs).sqrt() - beta_cs.sqrt()) / den - (gamma_cs / den).sqrt()
    m = m.with_columns(
        lv.alias("btc_logvol"),
        ((lv - lv.rolling_mean(1440)) / (lv.rolling_std(1440) + EPS)).alias("btc_volz_1440"),
        ((r.abs() / (pl.col(v) + 1e-8)).rolling_mean(60)).log1p().alias("btc_amihud_60"),
        ((pl.col(c) - vwap) / (vwap + EPS)).alias("btc_vwap_dev_60"),
        (2 * (alpha_cs.exp() - 1) / (1 + alpha_cs.exp())).clip(lower_bound=0.0)
        .rolling_mean(60).alias("btc_cs_spread_60"),
    )
    reg(["btc_logvol", "btc_volz_1440", "btc_amihud_60", "btc_vwap_dev_60",
         "btc_cs_spread_60"], "btc_volume")

    # ---- BTC momentum (capped at four) ----------------------------------
    d1 = pl.col(c).diff()
    gain = d1.clip(lower_bound=0.0).ewm_mean(alpha=1 / 14, adjust=False, min_samples=14)
    loss = (-d1).clip(lower_bound=0.0).ewm_mean(alpha=1 / 14, adjust=False, min_samples=14)
    ema12 = pl.col(c).ewm_mean(span=12, adjust=False)
    ema26 = pl.col(c).ewm_mean(span=26, adjust=False)
    macd = ema12 - ema26
    tr = pl.max_horizontal(pl.col(h) - pl.col(l),
                           (pl.col(h) - pl.col(c).shift(1)).abs(),
                           (pl.col(l) - pl.col(c).shift(1)).abs())
    ma60, sd60 = pl.col(c).rolling_mean(60), pl.col(c).rolling_std(60)
    m = m.with_columns(
        (100 - 100 / (1 + gain / (loss + EPS))).alias("btc_rsi_14"),
        ((macd - macd.ewm_mean(span=9, adjust=False)) / (pl.col(c) + EPS)).alias("btc_macd_hist"),
        (tr.ewm_mean(alpha=1 / 14, adjust=False, min_samples=14) / (pl.col(c) + EPS))
        .alias("btc_atr14_norm"),
        ((pl.col(c) - (ma60 - 2 * sd60)) / (4 * sd60 + EPS)).alias("btc_bb_pctb_60"),
    )
    reg(["btc_rsi_14", "btc_macd_hist", "btc_atr14_norm", "btc_bb_pctb_60"], "btc_momentum")

    # ---- exogenous blocks (already computed on their own clocks) --------
    reg(["gold_logret_1", "gold_logret_5", "gold_logret_60", "gold_logret_1440",
         "gold_rv_60", "gold_staleness", "gold_market_open", "gold_gap_decay"], "gold")
    reg(["xa_corr_60", "xa_corr_1440", "xa_beta_1440", "xa_spread_z",
         "xa_gold_leads_btc", "xa_btc_leads_gold"], "cross_asset")
    reg(["dxy_logret_1d", "dxy_logret_5d", "dxy_logret_21d", "dxy_pctrank_252",
         "dxy_age_days"], "dxy")
    reg([f"macro_pc{i + 1}" for i in range(K)]
        + ["macro_yield_curve", "macro_real_rate", "macro_m2_impulse",
           "macro_bs_impulse", "macro_age_days"], "macro")

    # ---- temporal --------------------------------------------------------
    tt = pl.col("t")
    mod = tt.dt.hour() * 60 + tt.dt.minute()
    dow, dom, moy = tt.dt.weekday() - 1, tt.dt.day() - 1, tt.dt.month() - 1
    cyc = []
    for name, expr, period in (("tod", mod, 1440.0), ("dow", dow, 7.0),
                               ("dom", dom, 31.0), ("moy", moy, 12.0)):
        cyc += [(2 * math.pi * expr / period).sin().alias(f"t_{name}_sin"),
                (2 * math.pi * expr / period).cos().alias(f"t_{name}_cos")]
    hr = tt.dt.hour()
    # minutes to / since the 13:30 UTC release slot on weekdays (CPI, NFP, PPI, retail sales)
    rel_h, rel_m = RELEASE_MINUTES_UTC["cpi_nfp"]
    rel_anchor = rel_h * 60 + rel_m
    delta = mod - rel_anchor
    m = m.with_columns(cyc + [
        ((hr >= 0) & (hr < 8)).cast(pl.Float32).alias("t_sess_asia"),
        ((hr >= 7) & (hr < 16)).cast(pl.Float32).alias("t_sess_europe"),
        ((hr >= 13) & (hr < 21)).cast(pl.Float32).alias("t_sess_us"),
        ((hr >= 13) & (hr < 16)).cast(pl.Float32).alias("t_sess_overlap"),
        (dow >= 5).cast(pl.Float32).alias("t_is_weekend"),
        pl.when(delta >= 0).then(delta).otherwise(delta + 1440).log1p().alias("t_since_release"),
        pl.when(delta <= 0).then(-delta).otherwise(1440 - delta).log1p().alias("t_to_release"),
        pl.col("btc_is_synthetic"),
    ])
    reg([f"t_{n}_{s}" for n in ("tod", "dow", "dom", "moy") for s in ("sin", "cos")]
        + ["t_sess_asia", "t_sess_europe", "t_sess_us", "t_sess_overlap", "t_is_weekend",
           "t_since_release", "t_to_release", "btc_is_synthetic"], "temporal")

    keep = [n for n in groups if groups[n] in cfg.blocks or n == "btc_is_synthetic"]
    # variate 0 must be the target's own series - see the section 7 note
    keep = ["btc_logret_1"] + [n for n in keep if n != "btc_logret_1"]
    return m.select(["t", *keep, "btc_close"]), {n: groups[n] for n in keep}


# ---- choose the fractional-differentiation order on the TRAIN split only ----
_lc_train = np.log(master.filter(pl.col("t") <= TRAIN_END_TS)["btc_close"].to_numpy())
_probe = _lc_train[-min(len(_lc_train), 400_000):]
FRAC_D, adf_table = None, []
for d in CFG.fracdiff_grid:
    stat = adf_stat(frac_diff(_probe, d, CFG.fracdiff_width))
    adf_table.append((d, stat))
    if FRAC_D is None and np.isfinite(stat) and stat < ADF_CRIT_95:
        FRAC_D = d
FRAC_D = FRAC_D if FRAC_D is not None else max(CFG.fracdiff_grid)
print("fractional differentiation - ADF t-statistic on the train split")
for d, s in adf_table:
    flag = "  <- selected" if d == FRAC_D else ("  (stationary)" if s < ADF_CRIT_95 else "")
    print(f"  d={d:.1f}   adf={s:8.3f}   crit95={ADF_CRIT_95}{flag}")
print(f"\nselected d = {FRAC_D}  (smallest order that rejects a unit root at 95%)")

In [ ]:
t0 = time.time()
feat_df, FEATURE_GROUP = build_features(master, CFG, FRAC_D)
print(f"features built in {time.time() - t0:.1f}s   shape {feat_df.shape}")

del master, gold_f, xa, dxy_f, macro_f, mac, mac_av, pca_frame
gc.collect()

FEATURE_NAMES_RAW = [c for c in feat_df.columns if c not in ("t", "btc_close")]
assert FEATURE_NAMES_RAW[0] == "btc_logret_1", "variate 0 must be the target series"

_counts: dict[str, int] = {}
for n in FEATURE_NAMES_RAW:
    _counts[FEATURE_GROUP[n]] = _counts.get(FEATURE_GROUP[n], 0) + 1
print(f"\n{len(FEATURE_NAMES_RAW)} raw features across {len(_counts)} blocks:")
for g, k in sorted(_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {g:<14} {k:>3}")

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 4px solid #48cae4; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🧼 §8 · Feature Hygiene</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;">Four operations, in this order, and every one of them is fitted on the <strong>training split alone</strong>. This is not a stylistic preference — fitting a scaler on the full dataset is a textbook leak that quietly inflates every downstream number.</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li><strong>Warm-up truncation.</strong> Rows before the longest lookback has filled are dropped outright rather than imputed.</li>
    <li><strong>Winsorisation at the 0.1% / 99.9% training quantiles</strong> — but <em>the target variate is exempt</em>. Clipping <code>btc_logret_1</code> would destroy exactly the tail events the model exists to handle, and would put the input variate on a different scale from the label sliced out of it. Robust losses handle the tails instead.</li>
    <li><strong>Collinearity pruning</strong> at |ρ| &gt; 0.98, keeping the earlier (simpler) feature of each pair. With attention running over variates, a duplicated variate does not just waste a slot — it splits attention mass across two copies of one signal.</li>
    <li><strong>Standardisation</strong> to zero mean, unit variance using training-split statistics, persisted to <code>scaler.json</code> and shipped in the export bundle.</li>
  </ul>
</div>

In [ ]:
t_all = feat_df["t"]
# Cast inside polars, before materialising. `to_numpy()` on Float64 columns builds a float64
# array first - 2.04 GB at the full profile - purely to be thrown away one line later. The
# matrix has always ended up float32; this only moves the cast earlier, where it is free.
X_raw = feat_df.select([pl.col(c).cast(pl.Float32) for c in FEATURE_NAMES_RAW]).to_numpy()
close_all = feat_df["btc_close"].to_numpy().astype(np.float64)   # 35 MB, stays float64
del feat_df
gc.collect()

# ---- 1. warm-up truncation ------------------------------------------------
finite_row = np.isfinite(X_raw).all(axis=1)
first_ok = int(np.argmax(finite_row))
assert finite_row[first_ok:].mean() > 0.99, "non-finite values persist well past the warm-up"
grid_start_idx = max(first_ok, int(np.searchsorted(
    t_all.to_numpy(), np.datetime64(ts(CFG.grid_start).replace(tzinfo=None), "us"))))
X_raw, close_all, t_all = X_raw[grid_start_idx:], close_all[grid_start_idx:], t_all[grid_start_idx:]
del finite_row
# any residual non-finite cell (e.g. a division edge) is zeroed and counted
_bad = ~np.isfinite(X_raw)
n_bad = int(_bad.sum())
X_raw[_bad] = 0.0
del _bad
gc.collect()
T = len(X_raw)
print(f"warm-up truncation: dropped {grid_start_idx:,} rows -> {T:,} remain "
      f"({t_all[0]} -> {t_all[-1]});  {n_bad:,} residual non-finite cells zeroed")

# `t_all` is sorted, so the training mask is a contiguous prefix and the train block can be a
# VIEW. `X_raw[train_row]` allocates a fresh ~0.8 GB copy and it is needed five times below.
train_row = (t_all <= TRAIN_END_TS).to_numpy()
n_tr = int(train_row.sum())
assert train_row[:n_tr].all() and not train_row[n_tr:].any(), \
    "train mask is not a contiguous prefix - the view used below would silently be wrong"
Xtr = X_raw[:n_tr]
print(f"train rows for fitting statistics: {n_tr:,} ({100 * n_tr / T:.1f}%)")

# ---- 2. winsorisation, target exempt --------------------------------------
TARGET_NAME = "btc_logret_1"
names = list(FEATURE_NAMES_RAW)
_q = np.quantile(Xtr, [CFG.winsor_q, 1 - CFG.winsor_q], axis=0)   # one pass, not two
lo_q, hi_q = _q[0].astype(np.float32), _q[1].astype(np.float32)
exempt = np.array([n == TARGET_NAME for n in names])
lo_q[exempt], hi_q[exempt] = -np.inf, np.inf
# a cell cannot be both below lo and above hi, so two counts sum to the union
n_clipped = int(np.count_nonzero(X_raw < lo_q) + np.count_nonzero(X_raw > hi_q))
np.clip(X_raw, lo_q, hi_q, out=X_raw)                 # in-place: no second full-size array
print(f"winsorised {n_clipped:,} cells ({100 * n_clipped / X_raw.size:.3f}%); "
      f"exempt: {[n for n, e in zip(names, exempt) if e]}")

# ---- 3. collinearity pruning ---------------------------------------------
sub = Xtr[:: max(1, n_tr // 200_000)]                 # correlation on a subsample is plenty
C = np.nan_to_num(np.corrcoef(sub, rowvar=False))
drop, kept = set(), []
for i, n in enumerate(names):
    if i == 0:                                        # never drop the target variate
        kept.append(i)
        continue
    dup = next((j for j in kept if abs(C[i, j]) > CFG.collinear_thresh), None)
    if dup is None:
        kept.append(i)
    else:
        drop.add(i)
        print(f"  drop {n:<24} |rho|={abs(C[i, dup]):.4f} vs {names[dup]}")
if len(kept) != X_raw.shape[1]:                       # column fancy-index copies; skip if no-op
    X_raw = X_raw[:, kept]
    Xtr = X_raw[:n_tr]
FEATURE_NAMES = [names[i] for i in kept]
del sub, C
gc.collect()
print(f"collinearity pruning: {len(names)} -> {len(FEATURE_NAMES)} features")

# ---- 4. standardisation, train split only ---------------------------------
# float64 accumulators over a float32 array: storage stays halved, the scaler stays accurate.
mu = Xtr.mean(axis=0, dtype=np.float64)
sd = Xtr.std(axis=0, dtype=np.float64)
sd[sd < 1e-8] = 1.0
X_raw -= mu.astype(np.float32)                        # in-place; `(X_raw - mu)` with a float64
X_raw /= sd.astype(np.float32)                        # mu would upcast the whole matrix
X = X_raw
del X_raw, Xtr
gc.collect()

N_VARIATES = X.shape[1]
TARGET_IDX = FEATURE_NAMES.index(TARGET_NAME)
assert TARGET_IDX == 0, "variate 0 must remain the target series after pruning"

# Scaled space -> raw log return. Every threshold quoted in basis points (the directional
# epsilon, the backtest thresholds) is a raw-return quantity; applying it to standardised
# values would make it ~1/SIGMA_TARGET times too small and it would stop filtering.
SIGMA_TARGET = float(sd[TARGET_IDX])
print(f"target sigma    {SIGMA_TARGET:.6e}   (1 bp = {1e-4 / SIGMA_TARGET:.4f} in scaled units)")

SCALER = {"mean": mu.tolist(), "std": sd.tolist(),
          "winsor_lo": np.where(np.isfinite(lo_q[kept]), lo_q[kept], None).tolist(),
          "winsor_hi": np.where(np.isfinite(hi_q[kept]), hi_q[kept], None).tolist(),
          "fitted_on": {"split": "train", "rows": n_tr, "t_max": str(t_all[n_tr - 1])}}

print(f"\nfeature matrix  X {X.shape}  {X.nbytes / 1024**3:.2f} GB  dtype={X.dtype}")
print(f"target variate  index {TARGET_IDX} = {TARGET_NAME}")
print(f"train-split standardisation check: mean {X[:n_tr].mean():+.2e}, "
      f"std {X[:n_tr].std():.4f}   (should be ~0 and ~1)")
print(f"val+test drift  : mean {X[n_tr:].mean():+.4f}, std {X[n_tr:].std():.4f} "
      f"  (drift here is real distribution shift, not a bug)")
print(f"\npeak RSS so far {peak_rss_gb():.2f} GB   "
      f"(this machine has 14.8 GB total; Kaggle has ~29 GB)")


<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 4px solid #00b4d8; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🚦 Causality gate — run before anything is frozen</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;"><strong>The one gate that belongs to feature construction rather than to training.</strong> It travels with <code>build_features</code> because it tests <code>build_features</code>; freezing a matrix that fails it would bake look-ahead into every session downstream.</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li><strong>Shift equivariance.</strong> Delaying the whole input by <em>k</em> minutes must delay every feature by exactly <em>k</em>.</li>
    <li><strong>Future-perturbation invariance</strong> — the stronger property. Corrupting the input from row <em>j</em> onward must leave every feature before <em>j</em> bit-identical. A centred window, a backward fill, or a reversed index fails this immediately.</li>
    <li>The remaining gates (split, scaler, overfit-a-batch, leakage) test the training side and run in <code>02_train.ipynb</code>.</li>
  </ul>
</div>

In [ ]:
GATES: dict[str, bool] = {}


def _synthetic_master(n: int, seed: int = 0) -> pl.DataFrame:
    """A synthetic master frame with the same column contract as the real one."""
    rng = np.random.default_rng(seed)
    t = pl.datetime_range(datetime(2022, 1, 1, tzinfo=UTC),
                          datetime(2022, 1, 1, tzinfo=UTC) + timedelta(minutes=n - 1),
                          interval="1m", time_zone="UTC", eager=True)
    close = 30_000 * np.exp(np.cumsum(rng.normal(0, 2e-4, n)))
    df = pl.DataFrame({
        "t": t, "btc_open": close, "btc_high": close * 1.0004, "btc_low": close * 0.9996,
        "btc_close": close, "btc_volume": rng.lognormal(0, 1, n), "btc_is_synthetic": np.zeros(n),
    })
    exo = ["gold_logret_1", "gold_logret_5", "gold_logret_60", "gold_logret_1440", "gold_rv_60",
           "gold_staleness", "gold_market_open", "gold_gap_decay", "xa_corr_60", "xa_corr_1440",
           "xa_beta_1440", "xa_spread_z", "xa_gold_leads_btc", "xa_btc_leads_gold",
           "dxy_logret_1d", "dxy_logret_5d", "dxy_logret_21d", "dxy_pctrank_252", "dxy_age_days",
           "macro_yield_curve", "macro_real_rate", "macro_m2_impulse", "macro_bs_impulse",
           "macro_age_days", *[f"macro_pc{i + 1}" for i in range(K)]]
    return df.with_columns([pl.Series(c, rng.normal(0, 1, n)) for c in exo])


def gate_shift_test(k: int = 37, n: int = 20_000) -> bool:
    """Two independent causality checks on build_features.

    1a) Shift equivariance - delaying the whole input by k minutes must delay every
        feature by exactly k (CLAUDE.md 7.6.1).
    1b) Future-perturbation invariance - the stronger property. Corrupting the input
        from row j onward must leave every feature at rows < j bit-identical. Any
        centred window, forward fill, or reversed index fails this immediately.
    """
    base = _synthetic_master(n)
    a, _ = build_features(base, CFG, FRAC_D)
    cols = [c for c in a.columns if c not in ("t", "btc_close") and not c.startswith("t_")]
    A = a.select(cols).to_numpy()

    # --- 1a: re-time the same values by +k minutes -------------------------
    b, _ = build_features(
        base.head(n - k).with_columns(pl.col("t") + pl.duration(minutes=k)), CFG, FRAC_D)
    B = b.select(cols).to_numpy()
    m = np.isfinite(A[: n - k]) & np.isfinite(B)
    err_shift = float(np.abs(A[: n - k] - B)[m].max()) if m.any() else np.inf
    ok_a = err_shift < 1e-8
    print(f"  1a shift equivariance      max|delta| = {err_shift:.3e}   "
          f"{'PASS' if ok_a else 'FAIL'}")

    # --- 1b: corrupt the future, inspect the past --------------------------
    j = n // 2
    fut = base.with_columns([
        pl.when(pl.int_range(pl.len()) >= j).then(pl.col(c) * 1.5 + 7.0).otherwise(pl.col(c)).alias(c)
        for c in base.columns if c != "t"
    ])
    c_, _ = build_features(fut, CFG, FRAC_D)
    Cm = c_.select(cols).to_numpy()[:j]
    m2 = np.isfinite(A[:j]) & np.isfinite(Cm)
    err_fut = float(np.abs(A[:j] - Cm)[m2].max()) if m2.any() else np.inf
    ok_b = err_fut < 1e-10
    print(f"  1b future-perturbation     max|delta| = {err_fut:.3e}   "
          f"{'PASS' if ok_b else 'FAIL'}   (past must not react to the future)")

    if not (ok_a and ok_b):
        bad = [c for i, c in enumerate(cols)
               if np.nanmax(np.abs(A[:j, i] - Cm[:, i])) > 1e-10]
        print(f"    LEAKING columns: {bad}")
    return bool(ok_a and ok_b)


print("Sanity gates")
GATES["shift"] = gate_shift_test()


<div style="background: linear-gradient(90deg, #2d0036, #4a0060); border-left: 4px solid #bf5af2; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">🧊 Freezing the feature matrix</h2>
  <p style="color: #c77dff; margin: 0 0 8px 0;">Everything above is deterministic given the raw files and <code>CFG</code>. Writing it down once turns that determinism into something checkable: two training sessions can now <em>prove</em> they used the same inputs instead of assuming it.</p>  <ul style="color: #c77dff; margin: 0; padding-left: 20px;">
    <li><code>features.npy</code> — <code>float32 (T, N)</code>, standardised, variate order fixed</li>
    <li><code>close.npy</code> — <code>float64 (T,)</code> raw close. Price reconstruction must never be derived from a standardised column</li>
    <li><code>timestamps.npy</code> — <code>int64</code> epoch-microseconds UTC. An integer cannot carry a timezone it forgot to declare</li>
    <li><code>scaler.json</code> — mean, std and winsorisation bounds, with the split they were fitted on recorded alongside them</li>
    <li><code>feature_manifest.json</code> — variate order, groups, target index, and the data-dependent choices (frac-diff order, PCA rank, gold offset) that cannot be recomputed without the raw data</li>
    <li><code>prep_metadata.json</code> — hashes of the raw files, of the matrix, of the manifest and of the scaler, plus the <strong>frozen config fields</strong> a training session must match</li>
  </ul>
</div>

In [ ]:
# ==========================================================================
#  The frozen-artifact contract.
#
#  This cell is byte-identical in 01_preprocess and 02_train. The producer and
#  the consumer must not be able to disagree about what a valid artifact is, so
#  the rules live in one cell that both notebooks carry rather than in two
#  descriptions that drift apart.
# ==========================================================================

# Config fields that shape the feature matrix itself. A training session that
# disagrees with the artifact on any of these is training on inputs it does not
# describe. `train_end` belongs here because the scaler is fitted on rows
# t <= train_end - changing it on the training side is a data leak, not a
# mismatch. `seq_len` belongs here because the warm-up truncation is computed as
# 1440 + seq_len + 60, so it decides which rows exist at all.
FROZEN_FIELDS = (
    "profile", "grid_start", "grid_end", "train_end", "val_end", "test_end",
    "seq_len", "pred_len", "blocks", "macro_n_pca", "fracdiff_grid",
    "fracdiff_width", "winsor_q", "collinear_thresh", "gold_utc_offset_h",
)

ARTIFACT_FILES = ("features.npy", "close.npy", "timestamps.npy",
                  "scaler.json", "feature_manifest.json", "prep_metadata.json")


def _sha256_file(p: Path, chunk: int = 1 << 23) -> str:
    """Hash a file in 8 MB chunks - `features.npy` is ~1 GB at the full profile."""
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for blk in iter(lambda: fh.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()


def _sha256_json(obj: dict) -> str:
    """Hash a JSON object by content, not by formatting, so indent changes are invisible."""
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()


def _frozen_of(cfg) -> dict:
    """Frozen fields with tuples normalised to lists, so a JSON round-trip is a no-op."""
    out: dict = {}
    for f in FROZEN_FIELDS:
        v = getattr(cfg, f)
        out[f] = list(v) if isinstance(v, tuple) else v
    return out


def artifact_dir(profile: str) -> Path:
    """One directory per profile, never shared: `tiny` and `full` are different matrices."""
    base = Path("/kaggle/working/processed") if ON_KAGGLE else Path("../data/processed")
    return base / f"features_{profile}"


def write_artifact(art_dir: Path, cfg, X: np.ndarray, close: np.ndarray,
                   t_us: np.ndarray, scaler: dict, manifest: dict,
                   raw_hashes: dict) -> dict:
    """Write the six artifact files and return the metadata that binds them together.

    `prep_metadata.json` is written last because it carries the hashes of the files
    written before it.
    """
    art_dir.mkdir(parents=True, exist_ok=True)
    assert X.dtype == np.float32, f"feature matrix must be float32, got {X.dtype}"
    assert close.dtype == np.float64, f"close must stay float64, got {close.dtype}"
    assert t_us.dtype == np.int64, f"timestamps must be int64 epoch-us, got {t_us.dtype}"
    assert len(X) == len(close) == len(t_us), (
        f"row counts disagree: X={len(X)}, close={len(close)}, t={len(t_us)}")

    np.save(art_dir / "features.npy", X)
    np.save(art_dir / "close.npy", close)
    np.save(art_dir / "timestamps.npy", t_us)
    (art_dir / "scaler.json").write_text(json.dumps(scaler, indent=2))
    (art_dir / "feature_manifest.json").write_text(json.dumps(manifest, indent=2))

    meta = {
        "created_utc": datetime.now(UTC).isoformat(),
        "profile": cfg.profile,
        "frozen": _frozen_of(cfg),
        "raw_sha256": raw_hashes,
        "features_sha256": _sha256_file(art_dir / "features.npy"),
        "manifest_sha256": _sha256_json(manifest),
        "scaler_sha256": _sha256_json(scaler),
        "shape": list(X.shape),
        "dtype": str(X.dtype),
        "t_first_us": int(t_us[0]),
        "t_last_us": int(t_us[-1]),
        "versions": {"python": sys.version.split()[0],
                     "numpy": np.__version__, "polars": pl.__version__},
        "peak_rss_gb": round(peak_rss_gb(), 3),
    }
    (art_dir / "prep_metadata.json").write_text(json.dumps(meta, indent=2))
    return meta


def load_artifact(art_dir: Path, cfg, mmap: bool = False) -> dict:
    """Load a frozen artifact, enforcing every rejection rule before returning.

    Each rule is printed first and asserted second. A bare AssertionError names the
    rule but not the state that broke it; printing the whole table means a rejected
    session says which value disagreed with which.

    `mmap=True` maps the matrix read-only, which is right for verification. Training
    needs `mmap=False`: the leakage gate overwrites the target column in place and
    restores it afterwards, and a read-only mapping would raise instead.
    """
    missing = [f for f in ARTIFACT_FILES if not (art_dir / f).exists()]
    assert not missing, (
        f"{art_dir} is not a frozen feature artifact - missing {missing}.\n"
        f"Run 01_preprocess.ipynb at this profile, or point at the Dataset holding it."
    )

    manifest = json.loads((art_dir / "feature_manifest.json").read_text())
    scaler = json.loads((art_dir / "scaler.json").read_text())
    meta = json.loads((art_dir / "prep_metadata.json").read_text())

    feat_hash = _sha256_file(art_dir / "features.npy")
    man_hash = _sha256_json(manifest)
    scl_hash = _sha256_json(scaler)

    t_us = np.load(art_dir / "timestamps.npy")
    close = np.load(art_dir / "close.npy")
    X = np.load(art_dir / "features.npy", mmap_mode="r" if mmap else None)

    want, got = _frozen_of(cfg), meta["frozen"]
    drift = {k: {"artifact": got.get(k), "session": want[k]}
             for k in want if got.get(k) != want[k]}
    n_feat = len(manifest["feature_order"])
    shape_ok = X.shape == (len(t_us), n_feat) and len(close) == len(t_us)
    tgt = manifest["feature_order"][manifest["target_index"]]

    checks = [
        ("1 features sha256", feat_hash == meta["features_sha256"],
         f"{feat_hash[:16]}... vs recorded {meta['features_sha256'][:16]}..."),
        ("2 manifest sha256", man_hash == meta["manifest_sha256"],
         f"{man_hash[:16]}... vs recorded {meta['manifest_sha256'][:16]}..."),
        ("3 frozen config", not drift,
         f"{len(FROZEN_FIELDS)} fields identical" if not drift else f"differs {drift}"),
        ("4 shape agreement", shape_ok,
         f"X{X.shape}  t={len(t_us):,}  close={len(close):,}  features={n_feat}"),
        ("5 target variate", tgt == "btc_logret_1",
         f"feature_order[{manifest['target_index']}] = {tgt}"),
        ("+ scaler sha256", scl_hash == meta.get("scaler_sha256"),
         f"{scl_hash[:16]}... vs recorded {str(meta.get('scaler_sha256'))[:16]}..."),
    ]

    print(f"artifact  {art_dir}")
    print(f"  built    {meta['created_utc']}  by numpy {meta['versions']['numpy']} / "
          f"polars {meta['versions']['polars']}")
    for name, ok, detail in checks:
        print(f"  {name:<20} {'PASS' if ok else 'FAIL'}   {detail}")
    for name, ok, detail in checks:
        assert ok, f"artifact REJECTED - rule {name}: {detail}"

    return {"X": X, "close": close, "t_us": t_us, "scaler": scaler,
            "manifest": manifest, "metadata": meta, "checks": checks,
            "features_sha256": feat_hash, "manifest_sha256": man_hash}


print(f"artifact contract loaded: {len(ARTIFACT_FILES)} files, "
      f"{len(FROZEN_FIELDS)} frozen config fields, 6 rejection rules")


In [ ]:
t0 = time.time()
ART_DIR = artifact_dir(CFG.profile)

# The raw-file hashes travel with the artifact. Without them a refreshed data vintage
# can be trained against a matrix built from the previous one, and nothing complains.
RAW_HASHES = {f: _sha256_file(RAW_DIR / f) for f in REQUIRED_FILES}
print(f"hashed {len(RAW_HASHES)} raw files in {time.time() - t0:.1f}s")

# Superset of the export manifest in section 16: everything that notebook needs in order
# to report the pipeline it trained on, without reopening the raw data.
PREP_MANIFEST = {
    "feature_order": FEATURE_NAMES,
    "n_variates": N_VARIATES,
    "target_index": TARGET_IDX,
    "target_name": TARGET_NAME,
    "seq_len": CFG.seq_len,
    "pred_len": CFG.pred_len,
    "groups": {n: FEATURE_GROUP.get(n, "other") for n in FEATURE_NAMES},
    "fracdiff_d": FRAC_D,
    "gold_utc_offset_hours": GOLD_OFFSET_H,
    "release_lag_months_days": {k: list(v) for k, v in RELEASE_LAG.items()},
    "dxy_lag_days": DXY_LAG_DAYS,
    "dropped_columns": sorted(MACRO_DROP),
    "macro_pca_components": int(K),
    "winsor_quantile": CFG.winsor_q,
    "collinearity_threshold": CFG.collinear_thresh,
    # ---- beyond the export manifest -------------------------------------
    "rows": int(T),
    "train_rows": int(n_tr),
    "sigma_target": SIGMA_TARGET,
    "raw_feature_count": len(FEATURE_NAMES_RAW),
    "pruned_features": sorted(set(FEATURE_NAMES_RAW) - set(FEATURE_NAMES)),
}

# int64 epoch-microseconds, not a string and not a timezone-aware object: the split
# boundaries downstream are integer comparisons, and an integer cannot carry a
# timezone it forgot to declare.
T_US = t_all.to_numpy().astype("datetime64[us]").astype(np.int64)

PREP_METADATA = write_artifact(ART_DIR, CFG, X, close_all, T_US,
                               SCALER, PREP_MANIFEST, RAW_HASHES)

print(f"\nartifact -> {ART_DIR.resolve()}")
_total = 0.0
for f in ARTIFACT_FILES:
    _mb = (ART_DIR / f).stat().st_size / 1024**2
    _total += _mb
    print(f"  {f:<24} {_mb:>10,.2f} MB")
print(f"  {'TOTAL':<24} {_total:>10,.2f} MB")

print(f"\nfeatures sha256  {PREP_METADATA['features_sha256']}")
print(f"manifest sha256  {PREP_METADATA['manifest_sha256']}")
print(f"scaler   sha256  {PREP_METADATA['scaler_sha256']}")
print(f"frozen fields    {', '.join(FROZEN_FIELDS)}")
print(f"\nwritten in {time.time() - t0:.1f}s   peak RSS {peak_rss_gb():.2f} GB")


<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 4px solid #00b4d8; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🔍 Verifying the frozen artifact</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;">The producer is checked by the consumer's own rules, in the producer's own session. An artifact that fails its own verification has not been produced — and learning that here costs seconds, whereas learning it on Kaggle costs a session.</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li>The six rejection rules <code>02_train.ipynb</code> applies, applied here first</li>
    <li><strong>Round-trip.</strong> <code>features.npy</code> re-read from disk must be bit-identical to the matrix in memory — mapped read-only, so this is a streaming comparison rather than a second full-size allocation</li>
    <li>Timestamps, raw close, and the scaler's fitted-row count all checked against what this session actually computed</li>
  </ul>
</div>

In [ ]:
# Verify the artifact the way 02_train will: re-read it from disk and apply every
# rejection rule. A producer that cannot pass its own consumer's checks has not
# produced anything usable, and finding that out here costs seconds rather than a
# Kaggle session.
_A = load_artifact(ART_DIR, CFG, mmap=True)

# Round-trip: the matrix on disk must be bit-identical to the one in memory. The
# mapping keeps this a streaming comparison instead of a second full-size copy.
GATES["artifact_roundtrip"] = bool(np.array_equal(_A["X"], X))
GATES["artifact_order"] = _A["manifest"]["feature_order"] == FEATURE_NAMES
GATES["artifact_time"] = bool(np.array_equal(_A["t_us"], T_US))
GATES["artifact_close"] = bool(np.array_equal(_A["close"], close_all, equal_nan=True))
GATES["artifact_scaler"] = _A["scaler"]["fitted_on"]["rows"] == n_tr

print()
print(f"  {'artifact_roundtrip':<20} {'PASS' if GATES['artifact_roundtrip'] else 'FAIL'}"
      f"   features.npy is bit-identical to X in memory")
print(f"  {'artifact_order':<20} {'PASS' if GATES['artifact_order'] else 'FAIL'}"
      f"   {len(FEATURE_NAMES)} variates in the order the model will receive them")
print(f"  {'artifact_time':<20} {'PASS' if GATES['artifact_time'] else 'FAIL'}"
      f"   {_A['t_us'][0]} -> {_A['t_us'][-1]} epoch-us")
print(f"  {'artifact_close':<20} {'PASS' if GATES['artifact_close'] else 'FAIL'}"
      f"   raw close preserved in float64 for price reconstruction")
print(f"  {'artifact_scaler':<20} {'PASS' if GATES['artifact_scaler'] else 'FAIL'}"
      f"   scaler fitted on {n_tr:,} train rows of {T:,}")

del _A
gc.collect()

print("\n" + "=" * 62)
for k, v in GATES.items():
    print(f"  {k:<20} {'PASS' if v else 'FAIL'}")
ALL_GATES_PASS = all(GATES.values())
print(f"  {'ALL GATES':<20} {'PASS' if ALL_GATES_PASS else 'FAIL'}")
print("=" * 62)
if not ALL_GATES_PASS:
    print("\n  Do NOT publish this artifact. A failing gate here means every number "
          "trained on it would be invalid.")
else:
    print(f"\n  Artifact frozen and verified. Next: 02_train.ipynb at "
          f"PROFILE = {CFG.profile!r}.")


<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); border-radius: 16px; padding: 28px 34px;">
  <h2 style="color: #e0aaff; margin: 0 0 12px 0;">✅ Next — hand the artifact to the GPU</h2>
  <ol style="color: #c8b6ff; margin: 0; padding-left: 22px; line-height: 1.7;">
    <li><strong>Confirm every gate printed PASS.</strong> A failing gate means the artifact is not publishable, whatever the numbers downstream look like.</li>
    <li><strong>Run <code>02_train.ipynb</code> at the same <code>PROFILE</code>.</strong> Locally it finds the artifact by itself; the frozen-field check rejects any mismatch.</li>
    <li><strong>For the <code>full</code> profile, publish once.</strong> Upload <code>data/processed/features_&lt;profile&gt;/</code> as a Kaggle Dataset, then attach it to every training session. Preprocessing never runs on Kaggle again, and every session trains on a matrix with the same hash.</li>
    <li><strong>Re-run this notebook only when the feature definitions change</strong> — and expect the hash to change with them. That is the mechanism working, not a fault.</li>
  </ol>
</div>